# 09. EXTENTIONS — הרחבות, הפקת תוצרים ואימות סופי

**????.** ???? ???? ??? ? ?'??? ????? ? ????? ?????

**מטרה.** מפרידה הרחבות ניסיוניות מן הליבה, מציגה builders ומריצה בדיקות שחזוריות וכיסוי.

**מקום בתהליך.** מחברת 9 מתוך 9 בסדרת המחקר. יש להריץ את המחברות לפי הסדר המופיע ב-`notebooks/README.md`; עם זאת, קוד האתחול מאתר את שורש הפרויקט גם בהרצה עצמאית.

**קלטים.** כל פלטי הפרויקט, קוד builders/validation ותיקיית extensions/.

**פלטים.** אימות סופי, כיסוי מלא של src, ומפת הרחבות מעבר לדרישות הבסיס.

**קשר לשאלות המחקר.** מבהירה מהו המימוש המרכזי ומהן הרחבות עתידיות, כנדרש בהנחיות הפרויקט.

**עקרונות שחזוריות.** `random_state` קבוע היכן שנדרש, כל התוצאות נשמרות תחת `outputs/`, מאפיינים מבוססי תווית נבנים מתוויות train בלבד, וכל קוד המקור הרלוונטי מוצג בתאי קוד עם נתיב ושורות מקור.

## אתחול בטוח ובלתי תלוי בתיקיית ההרצה

התא הבא מאתר את שורש הריפוזיטורי דרך `requirements.txt`, עובר אליו, מוסיף את `src` לנתיב הייבוא ומגדיר תצוגה עקבית. שינוי `__name__` מונע מבלוקי `if __name__ == "__main__"` שהועתקו כלשונם לפעול לפני תא ההרצה המתועד שלהם. הפלט הצפוי הוא הנתיב המוחלט של הפרויקט.

In [1]:
from pathlib import Path
import os
import sys
import warnings

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("לא נמצא שורש הפרויקט המכיל requirements.txt ו-data/")

ROOT = _find_project_root(Path.cwd())
os.chdir(ROOT)
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
__name__ = "research_notebook_09"
print(f"Project root: {ROOT}")

Project root: C:\Users\Jimi\Desktop\github-social-network


**פלט ומשמעות.** הופעת הנתיב `github-social-network` מאשרת שכל הנתיבים היחסיים בהמשך יופנו לפרויקט הנכון. אם התא נכשל, אין להמשיך: הכשל מצביע על הרצה מחוץ לעץ הפרויקט או על עותק חסר.

## מה נחשב EXTENTIONS בפרויקט הזה

הליבה המחקרית היא כל מה שרץ דרך `src/run_all.py` ומדווח ב-README ובדוח. תיקיית `extensions/` מכילה כיוונים מעבר לדרישות הבסיס: Label Propagation, GCN/GraphSAGE וגילוי תתי-תחומים בתוכן באמצעות SVD+KMeans. הם חשובים משום שהם בודקים message passing, semi-supervision ורזולוציית תוויות עשירה יותר; הם אינם מעורבבים בטבלאות הליבה, מפני שחלקם דורשים PyTorch/תקציב ניסוי נוסף וחלקם מחוץ להיקף האלגוריתמים שנבחר להגשה. ההפרדה מונעת הצגת תוצאה לא מאומתת כטענת פרויקט.

שיפורים עתידיים מוצעים: nested cross-validation להיפרפרמטרים, יותר rewires לשיפור רזולוציית p, בדיקה טמפורלית אמיתית ל-link prediction, calibration למדדי הסתברות ובדיקת fairness מעבר לתווית הבינארית.

## מלאי הרחבות קיים

התא קורא את README של ההרחבות ומציג את קובצי Python בלי להפעיל ניסויים שאינם חלק מהצנרת המאומתת. הפלט הצפוי הוא ארבעה קבצים ותיאור ההפרדה מהליבה.

In [2]:
import pandas as pd

extension_files = sorted((ROOT / "extensions").glob("*.py"))
extension_inventory = pd.DataFrame([
    {"file": p.name, "lines": len(p.read_text(encoding="utf-8").splitlines()), "bytes": p.stat().st_size}
    for p in extension_files
])
display(extension_inventory)
print((ROOT / "extensions" / "README.md").read_text(encoding="utf-8"))

,file,lines,bytes
0,gnn.py,187,7221
1,label_propagation.py,61,2107
2,run_gnn.py,221,9325
3,run_subfields.py,191,7978


# Optional extensions

This directory preserves exploratory implementations that are **not** part of
the final course-only project, its reproducibility pipeline, its reported
results, or its grading claims.

- `gnn.py` and `run_gnn.py`: GCN and GraphSAGE experiments.
- `run_subfields.py`: TruncatedSVD/K-means subfield exploration with
  Silhouette, NMI, and ARI evaluation.
- `label_propagation.py`: the former label-propagation classifier.

The authoritative project code lives in `src/`. Running `python src/run_all.py`
does not import or execute anything in this directory.



**פלט ומשמעות.** ההרחבות שמורות ונגישות, אך אינן משנות את תשובות RQ1-RQ4. זוהי הבחנה מפורשת בין contribution מרכזי לבין exploration עתידי.

## קוד המקור: `src/final_validation.py`

בדיקות fail-fast לתוצרים, פרוטוקולים, טווחי metrics ודוח סופי.

לפני טעינת הקטעים נגדיר עבורם `__file__` וירטואלי התואם למיקומם המקורי. הדבר משמר את לוגיקת חישוב `PROJECT_DIR` בלי לשנות שורה בקוד המקור. SHA-256 של המקור: `b8b4c2c21c88590c75e1109f5627558ac5677cda4b61c679369a11e51d141d55`.

התא הבא מכין את הקשר הנתיבים עבור קובץ המקור. הוא אינו משנה את הקובץ; הפלט הצפוי הוא נתיב המקור שממנו הועתק הקוד.

In [3]:
__file__ = str(ROOT / 'src/final_validation.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\final_validation.py


**פלט ומשמעות.** הנתיב המודפס מאפשר לבקר שהקבועים בקוד יחושבו ביחס לריפוזיטורי הנוכחי בלבד.

### final_validation.py, שורות 1-100: ייבואים, קבועים והגדרות מודול

**מה ומדוע.** הבלוק מרכז את התלויות, הקבועים והנחות ברירת המחדל של המודול; בלעדיו הפונקציות הבאות אינן עצמאיות.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [4]:
# -*- coding: utf-8 -*-
"""Fail-fast final-project validation used by both CLI and notebook."""

from pathlib import Path
import json
import math
import re
from typing import Any
import zipfile

import networkx as nx
import numpy as np
import pandas as pd
from pypdf import PdfReader

from data_loader import load_dataset
from link_prediction import sample_link_prediction_split


SEEDS = (42, 7, 21, 84, 123)
NOTEBOOK_RELATIVE_PATH = Path(
    "notebooks/GitHub_Developer_Field_Prediction_Network_Only.ipynb"
)
REQUIRED_TABLES = (
    "data_quality_checks.csv",
    "network_basic_stats.csv",
    "label_distribution.csv",
    "edge_type_counts.csv",
    "centrality_top_developers.csv",
    "centrality_top_overlap.csv",
    "community_summary.csv",
    "community_metrics.csv",
    "classification_results.csv",
    "classification_summary.csv",
    "threshold_tuning_results.csv",
    "threshold_tuning_summary.csv",
    "ablation_model_results.csv",
    "ablation_model_results_summary.csv",
    "embedding_classification_results.csv",
    "embedding_classification_summary.csv",
    "edge_reconstruction_results.csv",
    "edge_reconstruction_summary.csv",
    "edge_reconstruction_by_type.csv",
    "edge_reconstruction_by_type_summary.csv",
    "edge_reconstruction_same_vs_cross.csv",
    "edge_reconstruction_same_vs_cross_summary.csv",
    "edge_reconstruction_same_vs_cross_gaps.csv",
    "edge_split_diagnostics.csv",
    "null_model_homophily.csv",
    "null_model_modularity.csv",
    "network_vs_random_models.csv",
    "powerlaw_fit.csv",
)
REQUIRED_FIGURES = (
    "label_distribution.png",
    "degree_distribution.png",
    "edge_type_distribution.png",
    "sampled_network_by_label.png",
    "community_size_distribution.png",
    "community_size_and_composition.png",
    "classification_comparison.png",
    "threshold_tradeoff.png",
    "ablation_f1_auc_comparison.png",
    "representation_comparison.png",
    "embedding_tsne.png",
    "edge_reconstruction_auc.png",
    "edge_reconstruction_same_vs_cross.png",
    "homophily_null_distribution.png",
    "degree_distribution_loglog.png",
)
REQUIRED_EMBEDDINGS = ("deepwalk_64.npz", "node2vec_64.npz")

EXPECTED_CLASSIFICATION_MODELS = {
    "Majority Baseline",
    "Neighbor Majority",
    "Logistic Regression - Graph Features Only",
    "Logistic Regression - Graph + Community Features",
}
EXPECTED_ABLATION_MODELS = {
    "Logistic Regression - Content Features Only",
    "Logistic Regression - Network Only (Graph + Community)",
    "Logistic Regression - Hybrid (Network + Content)",
}
EXPECTED_EMBEDDING_MODELS = {
    "Logistic Regression - DeepWalk",
    "Logistic Regression - DeepWalk + Graph + Community",
    "Logistic Regression - Node2Vec",
    "Logistic Regression - Node2Vec + Graph + Community",
}
FORBIDDEN_PUBLIC_PATTERNS = (
    r"\bGCN\b",
    r"\bGraphSAGE\b",
    r"\bLabel\s+Propagation\b",
    r"\bTruncatedSVD\b",
    r"\bNMI\b",
    r"\bARI\b",
    r"\bSilhouette\b",
)




**פלט ומשמעות.** בלוק `ייבואים, קבועים והגדרות מודול` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 101-105: `_assert`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [5]:
def _assert(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)




**פלט ומשמעות.** בלוק ``_assert`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 106-178: `collect_key_results`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי. תיעוד המקור מסביר: Collect the exact numerical claims embedded in public artifacts.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [6]:
def collect_key_results(project_dir: Path) -> dict[str, Any]:
    """Collect the exact numerical claims embedded in public artifacts."""
    tables_dir = project_dir / "outputs" / "tables"
    basic = pd.read_csv(tables_dir / "network_basic_stats.csv").iloc[0]
    classification = pd.read_csv(
        tables_dir / "classification_summary.csv"
    )
    best_classification = classification.loc[
        classification["roc_auc_mean"].idxmax()
    ]
    embeddings = pd.read_csv(
        tables_dir / "embedding_classification_summary.csv"
    ).set_index("model")
    reconstruction = pd.read_csv(
        tables_dir / "edge_reconstruction_summary.csv"
    )
    best_reconstruction = reconstruction.loc[
        reconstruction["roc_auc_mean"].idxmax()
    ]
    groups = pd.read_csv(
        tables_dir / "edge_reconstruction_same_vs_cross_summary.csv"
    )
    best_groups = groups[
        groups["heuristic"] == best_reconstruction["heuristic"]
    ].set_index("edge_group")
    homophily_null = pd.read_csv(
        tables_dir / "null_model_homophily.csv"
    ).set_index("null_model")
    modularity_null = pd.read_csv(
        tables_dir / "null_model_modularity.csv"
    ).iloc[0]
    return {
        "network_nodes": int(basic["num_nodes"]),
        "network_edges": int(basic["num_edges"]),
        "homophily_rate": float(basic["homophily_rate"]),
        "best_classification_model": str(best_classification["model"]),
        "best_classification_roc_auc": float(
            best_classification["roc_auc_mean"]
        ),
        "deepwalk_roc_auc": float(
            embeddings.loc[
                "Logistic Regression - DeepWalk", "roc_auc_mean"
            ]
        ),
        "node2vec_roc_auc": float(
            embeddings.loc[
                "Logistic Regression - Node2Vec", "roc_auc_mean"
            ]
        ),
        "best_reconstruction_heuristic": str(
            best_reconstruction["heuristic"]
        ),
        "best_reconstruction_roc_auc": float(
            best_reconstruction["roc_auc_mean"]
        ),
        "best_reconstruction_same_field_roc_auc": float(
            best_groups.loc["same-field", "roc_auc_mean"]
        ),
        "best_reconstruction_cross_field_roc_auc": float(
            best_groups.loc["cross-field", "roc_auc_mean"]
        ),
        "label_shuffle_empirical_p": float(
            homophily_null.loc["label_shuffle", "empirical_p"]
        ),
        "degree_rewire_empirical_p": float(
            homophily_null.loc[
                "degree_preserving_rewire", "empirical_p"
            ]
        ),
        "modularity_empirical_p": float(modularity_null["empirical_p"]),
    }




**פלט ומשמעות.** בלוק ``collect_key_results`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 179-190: `_check_required_files`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [7]:
def _check_required_files(project_dir: Path) -> None:
    for filename in REQUIRED_TABLES:
        path = project_dir / "outputs" / "tables" / filename
        _assert(path.is_file() and path.stat().st_size > 0, f"Missing table: {path}")
    for filename in REQUIRED_FIGURES:
        path = project_dir / "outputs" / "figures" / filename
        _assert(path.is_file() and path.stat().st_size > 0, f"Missing figure: {path}")
    for filename in REQUIRED_EMBEDDINGS:
        path = project_dir / "outputs" / "embeddings" / filename
        _assert(path.is_file() and path.stat().st_size > 0, f"Missing embedding: {path}")




**פלט ומשמעות.** בלוק ``_check_required_files`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 191-237: `_check_csvs`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [8]:
def _check_csvs(project_dir: Path) -> None:
    tables_dir = project_dir / "outputs" / "tables"
    for filename in REQUIRED_TABLES:
        frame = pd.read_csv(tables_dir / filename)
        _assert(not frame.empty, f"CSV is empty: {filename}")
        numeric = frame.select_dtypes(include=[np.number])
        _assert(
            np.isfinite(numeric.to_numpy(dtype=float)).all(),
            f"NaN or Inf found in {filename}",
        )
        for column in numeric.columns:
            lower = column.lower()
            values = numeric[column].to_numpy(dtype=float)
            is_gap = "gap" in lower or "_minus_" in lower
            if is_gap:
                _assert(
                    ((-1 <= values) & (values <= 1)).all(),
                    f"Out-of-range [-1,1] gap {column} in {filename}",
                )
            elif lower.startswith("loglik_ratio"):
                # Likelihood-ratio statistics are signed and unbounded.
                continue
            elif any(
                token in lower
                for token in (
                    "roc_auc",
                    "average_precision",
                    "accuracy",
                    "precision",
                    "recall",
                    "f1",
                    "purity",
                    "ratio",
                    "fraction",
                )
            ):
                _assert(
                    ((0 <= values) & (values <= 1)).all(),
                    f"Out-of-range [0,1] metric {column} in {filename}",
                )
            if "modularity" in lower or "assortativity" in lower:
                _assert(
                    ((-1 <= values) & (values <= 1)).all(),
                    f"Out-of-range [-1,1] metric {column} in {filename}",
                )




**פלט ומשמעות.** בלוק ``_check_csvs`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 238-281: `_check_graph_and_splits`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [9]:
def _check_graph_and_splits(project_dir: Path) -> None:
    G, edges_df, labels_df, labels_dict = load_dataset(project_dir / "data")
    _assert(G.number_of_nodes() == 37_700, "Unexpected node count.")
    _assert(G.number_of_edges() == 289_003, "Unexpected edge count.")
    _assert(nx.is_connected(G), "Observed graph is not connected.")
    _assert(nx.number_of_selfloops(G) == 0, "Self-loops found.")
    _assert(
        len(edges_df) == G.number_of_edges(),
        "Duplicate undirected edge rows found.",
    )
    _assert(not labels_df["id"].duplicated().any(), "Duplicate label IDs found.")
    _assert(not labels_df["label"].isna().any(), "Missing labels found.")
    _assert(set(G.nodes()) == set(labels_df["id"]), "Graph/label node mismatch.")

    diagnostics = pd.read_csv(
        project_dir / "outputs" / "tables" / "edge_split_diagnostics.csv"
    ).set_index("seed")
    _assert(set(diagnostics.index.astype(int)) == set(SEEDS), "Split seeds mismatch.")
    _assert(diagnostics["train_connected"].astype(bool).all(), "Disconnected train split.")
    _assert((diagnostics["isolated_nodes"] == 0).all(), "Isolates found in train split.")
    for edge_type in ("web_web", "ml_ml", "web_ml"):
        _assert(
            (
                diagnostics[f"positive_{edge_type}"]
                == diagnostics[f"negative_{edge_type}"]
            ).all(),
            f"Unmatched negative counts for {edge_type}.",
        )

    # Recompute the split invariant rather than trusting the diagnostic CSV.
    for seed in SEEDS:
        train_graph, _, _, split_diagnostics = sample_link_prediction_split(
            G,
            labels_dict,
            test_fraction=0.10,
            seed=seed,
        )
        _assert(nx.is_connected(train_graph), f"Seed {seed} disconnected.")
        _assert(
            split_diagnostics["isolated_nodes"] == 0,
            f"Seed {seed} created isolates.",
        )




**פלט ומשמעות.** בלוק ``_check_graph_and_splits`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 282-298: `_check_empirical_p_values`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [10]:
def _check_empirical_p_values(project_dir: Path) -> None:
    tables_dir = project_dir / "outputs" / "tables"
    homophily = pd.read_csv(tables_dir / "null_model_homophily.csv")
    modularity = pd.read_csv(tables_dir / "null_model_modularity.csv")
    p_values = pd.concat(
        [homophily["empirical_p"], modularity["empirical_p"]],
        ignore_index=True,
    )
    _assert(((p_values > 0) & (p_values <= 1)).all(), "Invalid empirical p-value.")
    for _, row in homophily.iterrows():
        minimum = 1.0 / (int(row["n_samples"]) + 1)
        _assert(
            row["empirical_p"] + 1e-15 >= minimum,
            "Empirical p-value violates the finite-simulation correction.",
        )




**פלט ומשמעות.** בלוק ``_check_empirical_p_values`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 299-330: `_check_model_sets`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [11]:
def _check_model_sets(project_dir: Path) -> None:
    tables_dir = project_dir / "outputs" / "tables"
    classification = set(
        pd.read_csv(tables_dir / "classification_summary.csv")["model"]
    )
    ablation = set(
        pd.read_csv(tables_dir / "ablation_model_results_summary.csv")["model"]
    )
    embeddings = set(
        pd.read_csv(tables_dir / "embedding_classification_summary.csv")["model"]
    )
    _assert(classification == EXPECTED_CLASSIFICATION_MODELS, "Classification models mismatch.")
    _assert(ablation == EXPECTED_ABLATION_MODELS, "Ablation models mismatch.")
    _assert(embeddings == EXPECTED_EMBEDDING_MODELS, "Embedding models mismatch.")

    logical_models = set(classification) | set(embeddings) | set(ablation)
    logical_models.discard(
        "Logistic Regression - Network Only (Graph + Community)"
    )
    _assert(len(logical_models) == 10, f"Expected 10 logical models, got {len(logical_models)}.")

    forbidden_filenames = re.compile(
        r"(gnn|subfield|label[_ -]?propagation)", re.IGNORECASE
    )
    for path in (project_dir / "outputs").rglob("*"):
        if path.is_file():
            _assert(
                not forbidden_filenames.search(path.name),
                f"Removed-model output remains: {path.name}",
            )




**פלט ומשמעות.** בלוק ``_check_model_sets`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 331-339: `_docx_text`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [12]:
def _docx_text(path: Path) -> str:
    with zipfile.ZipFile(path) as archive:
        chunks = []
        for name in archive.namelist():
            if name.startswith("word/") and name.endswith(".xml"):
                chunks.append(archive.read(name).decode("utf-8", errors="ignore"))
        return "\n".join(chunks)




**פלט ומשמעות.** בלוק ``_docx_text`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 340-370: `_check_public_artifacts`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [13]:
def _check_public_artifacts(project_dir: Path) -> None:
    notebook_path = project_dir / NOTEBOOK_RELATIVE_PATH
    report_source = project_dir / "report" / "report_source.docx"
    report_pdf = project_dir / "report" / "final_report.pdf"
    _assert(notebook_path.is_file(), "Notebook is missing.")
    _assert(report_source.is_file(), "Editable report source is missing.")
    _assert(report_pdf.is_file(), "Final report PDF is missing.")
    pdf_reader = PdfReader(str(report_pdf))
    _assert(1 <= len(pdf_reader.pages) <= 8, "Final report exceeds eight pages.")

    public_texts = {
        "README.md": (project_dir / "README.md").read_text(encoding="utf-8"),
        "notebook": notebook_path.read_text(encoding="utf-8"),
        "report_source.docx": _docx_text(report_source),
        "final_report.pdf": "\n".join(
            page.extract_text() or "" for page in pdf_reader.pages
        ),
    }
    for label, text in public_texts.items():
        for pattern in FORBIDDEN_PUBLIC_PATTERNS:
            _assert(
                re.search(pattern, text, flags=re.IGNORECASE) is None,
                f"Forbidden method remains in {label}: {pattern}",
            )
        _assert(
            re.search(r"חיזוי\s+עתידי|future\s+link", text, flags=re.IGNORECASE)
            is None,
            f"Future-prediction wording remains in {label}.",
        )




**פלט ומשמעות.** בלוק ``_check_public_artifacts`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 371-416: `_check_notebook`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [14]:
def _check_notebook(
    project_dir: Path, verify_execution: bool
) -> None:
    notebook_path = project_dir / NOTEBOOK_RELATIVE_PATH
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    embedded = notebook.get("metadata", {}).get(
        "final_project_expected_results"
    )
    _assert(isinstance(embedded, dict) and embedded, "Notebook expected-results metadata missing.")
    actual = collect_key_results(project_dir)
    _assert(set(embedded) == set(actual), "Notebook result keys do not match CSV result keys.")
    for key, expected in embedded.items():
        observed = actual[key]
        if isinstance(expected, (int, float)) and isinstance(observed, (int, float)):
            _assert(
                math.isclose(float(expected), float(observed), rel_tol=1e-7, abs_tol=1e-9),
                f"Notebook/CSV mismatch for {key}: {expected} != {observed}",
            )
        else:
            _assert(expected == observed, f"Notebook/CSV mismatch for {key}.")

    if verify_execution:
        code_cells = [
            cell for cell in notebook["cells"] if cell.get("cell_type") == "code"
        ]
        _assert(code_cells, "Notebook has no code cells.")
        _assert(
            all(cell.get("execution_count") is not None for cell in code_cells),
            "Notebook contains unexecuted code cells.",
        )
        errors = [
            output
            for cell in code_cells
            for output in cell.get("outputs", [])
            if output.get("output_type") == "error"
        ]
        _assert(not errors, "Notebook contains error outputs.")
        final_text = json.dumps(
            code_cells[-1].get("outputs", []), ensure_ascii=False
        )
        _assert(
            "ALL FINAL PROJECT VALIDATIONS PASSED" in final_text,
            "Notebook final validation success message is missing.",
        )




**פלט ומשמעות.** בלוק ``_check_notebook`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 417-437: `run_all_validations`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי. תיעוד המקור מסביר: Run every final-project validation and print success only at the end.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [15]:
def run_all_validations(
    project_dir: str | Path | None = None,
    *,
    verify_notebook_execution: bool = True,
) -> None:
    """Run every final-project validation and print success only at the end."""
    root = (
        Path(project_dir).resolve()
        if project_dir is not None
        else Path(__file__).resolve().parent.parent
    )
    _check_required_files(root)
    _check_csvs(root)
    _check_graph_and_splits(root)
    _check_empirical_p_values(root)
    _check_model_sets(root)
    _check_public_artifacts(root)
    _check_notebook(root, verify_notebook_execution)
    print("ALL FINAL PROJECT VALIDATIONS PASSED")




**פלט ומשמעות.** בלוק ``run_all_validations`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### final_validation.py, שורות 438-439: `__main__ guard`

**מה ומדוע.** זהו שער ההרצה המקורי של קובץ CLI. במחברת הוא נשמר כלשונו, אך אינו מופעל כי `__name__` הוגדר כשם מחברת; ההרצה נעשית בתא מפורש ומתועד.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** אין פלט, משום ששער ה-CLI מושבת בכוונה בסביבת המחברת.

In [16]:
if __name__ == "__main__":
    run_all_validations()


**פלט ומשמעות.** בלוק ``__main__ guard`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

In [17]:
_run_all_validations = run_all_validations

## קוד המקור: `src/run_all.py`

התזמור הסמכותי של כל הצנרת, מניקוי outputs ועד דוחות ואימות.

לפני טעינת הקטעים נגדיר עבורם `__file__` וירטואלי התואם למיקומם המקורי. הדבר משמר את לוגיקת חישוב `PROJECT_DIR` בלי לשנות שורה בקוד המקור. SHA-256 של המקור: `730bc482206c075746e5c649bd2053ec44055ab6b7905966093919e804d83c79`.

התא הבא מכין את הקשר הנתיבים עבור קובץ המקור. הוא אינו משנה את הקובץ; הפלט הצפוי הוא נתיב המקור שממנו הועתק הקוד.

In [18]:
__file__ = str(ROOT / 'src/run_all.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\run_all.py


**פלט ומשמעות.** הנתיב המודפס מאפשר לבקר שהקבועים בקוד יחושבו ביחס לריפוזיטורי הנוכחי בלבד.

### run_all.py, שורות 1-23: ייבואים, קבועים והגדרות מודול

**מה ומדוע.** הבלוק מרכז את התלויות, הקבועים והנחות ברירת המחדל של המודול; בלעדיו הפונקציות הבאות אינן עצמאיות.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [19]:
# -*- coding: utf-8 -*-
"""Rebuild every experiment, public output, notebook source and report source."""

import argparse
import gc
from pathlib import Path
import random
import shutil

import numpy as np

from run_ablation import main as run_ablation
from run_classification import main as run_classification
from run_core_analysis import main as run_core_analysis
from run_embeddings import main as run_embeddings
from run_link_prediction import main as run_edge_reconstruction
from run_null_models import main as run_null_models


PROJECT_DIR = Path(__file__).resolve().parent.parent
OUTPUTS_DIR = PROJECT_DIR / "outputs"




**פלט ומשמעות.** בלוק `ייבואים, קבועים והגדרות מודול` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### run_all.py, שורות 24-36: `_clean_outputs`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי. תיעוד המקור מסביר: Delete only the resolved project outputs directory.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [20]:
def _clean_outputs() -> None:
    """Delete only the resolved project outputs directory."""
    root = PROJECT_DIR.resolve()
    target = OUTPUTS_DIR.resolve()
    if target.parent != root or target.name != "outputs":
        raise RuntimeError(f"Refusing to clean unexpected path: {target}")
    if target.exists():
        shutil.rmtree(target)
    (target / "tables").mkdir(parents=True, exist_ok=True)
    (target / "figures").mkdir(parents=True, exist_ok=True)
    (target / "embeddings").mkdir(parents=True, exist_ok=True)




**פלט ומשמעות.** בלוק ``_clean_outputs`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### run_all.py, שורות 37-74: `main`

**מה ומדוע.** פונקציית התזמור של הקובץ. היא מוגדרת כאן אך אינה מופעלת אוטומטית; תא הרצה מפורש בהמשך מפעיל אותה ושומר פלטים.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [21]:
def main(clean: bool = True) -> None:
    random.seed(42)
    np.random.seed(42)
    if clean:
        _clean_outputs()

    stages = (
        ("core analysis", run_core_analysis),
        ("node classification", run_classification),
        ("content/network ablation", run_ablation),
        ("DeepWalk and Node2Vec", run_embeddings),
        ("connected edge reconstruction", run_edge_reconstruction),
        ("null and network models", run_null_models),
    )
    for name, function in stages:
        print(f"\n=== {name} ===")
        function()
        gc.collect()

    from build_notebook import main as build_notebook
    from build_readme import main as build_readme
    from build_report import main as build_report
    from build_report_pdf import main as build_report_pdf

    print("\n=== notebook source ===")
    build_notebook()
    print("\n=== README ===")
    build_readme()
    print("\n=== editable report source ===")
    build_report()
    print("\n=== final report PDF ===")
    build_report_pdf()
    print(
        "\nExperiments and public artifacts rebuilt. Execute the notebook, then "
        "run python src/final_validation.py."
    )




**פלט ומשמעות.** בלוק ``main`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### run_all.py, שורות 75-83: `__main__ guard`

**מה ומדוע.** זהו שער ההרצה המקורי של קובץ CLI. במחברת הוא נשמר כלשונו, אך אינו מופעל כי `__name__` הוגדר כשם מחברת; ההרצה נעשית בתא מפורש ומתועד.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** אין פלט, משום ששער ה-CLI מושבת בכוונה בסביבת המחברת.

In [22]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--keep-existing",
        action="store_true",
        help="Do not remove old output files before rebuilding.",
    )
    args = parser.parse_args()
    main(clean=not args.keep_existing)


**פלט ומשמעות.** בלוק ``__main__ guard`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

## קוד המקור: `src/build_notebook.py`

ה-builder ההיסטורי למחברת היחידה; נשמר במלואו לצורך עקיבות לאחור.

לפני טעינת הקטעים נגדיר עבורם `__file__` וירטואלי התואם למיקומם המקורי. הדבר משמר את לוגיקת חישוב `PROJECT_DIR` בלי לשנות שורה בקוד המקור. SHA-256 של המקור: `c44cef11c7b0de469e32e21e7623ff32969519f191d26dd37f7cc24089a0af78`.

התא הבא מכין את הקשר הנתיבים עבור קובץ המקור. הוא אינו משנה את הקובץ; הפלט הצפוי הוא נתיב המקור שממנו הועתק הקוד.

In [23]:
__file__ = str(ROOT / 'src/build_notebook.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\build_notebook.py


**פלט ומשמעות.** הנתיב המודפס מאפשר לבקר שהקבועים בקוד יחושבו ביחס לריפוזיטורי הנוכחי בלבד.

### build_notebook.py, שורות 1-24: ייבואים, קבועים והגדרות מודול

**מה ומדוע.** הבלוק מרכז את התלויות, הקבועים והנחות ברירת המחדל של המודול; בלעדיו הפונקציות הבאות אינן עצמאיות.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [24]:
# -*- coding: utf-8 -*-
"""Build the final, ordered notebook from freshly generated CSV results."""

from pathlib import Path
import sys

import nbformat as nbf
import pandas as pd

SRC_DIR = Path(__file__).resolve().parent
PROJECT_DIR = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))

from final_validation import collect_key_results


NOTEBOOK_PATH = (
    PROJECT_DIR
    / "notebooks"
    / "GitHub_Developer_Field_Prediction_Network_Only.ipynb"
)
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"




**פלט ומשמעות.** בלוק `ייבואים, קבועים והגדרות מודול` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 25-28: `_f`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [25]:
def _f(value: float, digits: int = 3) -> str:
    return f"{float(value):.{digits}f}"




**פלט ומשמעות.** בלוק ``_f`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 29-40: `_table_rows`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [26]:
def _table_rows(frame: pd.DataFrame, columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    rule = "|" + "|".join("---" for _ in columns) + "|"
    rows = [
        "| "
        + " | ".join(str(row[column]) for column in columns)
        + " |"
        for _, row in frame.iterrows()
    ]
    return "\n".join([header, rule, *rows])




**פלט ומשמעות.** בלוק ``_table_rows`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 41-82: `_result_context`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [27]:
def _result_context() -> dict:
    key = collect_key_results(PROJECT_DIR)
    centrality = pd.read_csv(
        TABLES_DIR / "centrality_top_developers.csv"
    )
    overlap = pd.read_csv(TABLES_DIR / "centrality_top_overlap.csv")
    community_metrics = pd.read_csv(
        TABLES_DIR / "community_metrics.csv"
    ).iloc[0]
    communities = pd.read_csv(TABLES_DIR / "community_summary.csv")
    classification = pd.read_csv(
        TABLES_DIR / "classification_summary.csv"
    )
    threshold = pd.read_csv(TABLES_DIR / "threshold_tuning_summary.csv")
    ablation = pd.read_csv(
        TABLES_DIR / "ablation_model_results_summary.csv"
    )
    embeddings = pd.read_csv(
        TABLES_DIR / "embedding_classification_summary.csv"
    )
    reconstruction = pd.read_csv(
        TABLES_DIR / "edge_reconstruction_summary.csv"
    )
    reconstruction_groups = pd.read_csv(
        TABLES_DIR / "edge_reconstruction_same_vs_cross_summary.csv"
    )
    reconstruction_gaps = pd.read_csv(
        TABLES_DIR / "edge_reconstruction_same_vs_cross_gaps.csv"
    )
    null_homophily = pd.read_csv(
        TABLES_DIR / "null_model_homophily.csv"
    )
    null_modularity = pd.read_csv(
        TABLES_DIR / "null_model_modularity.csv"
    ).iloc[0]
    networks = pd.read_csv(
        TABLES_DIR / "network_vs_random_models.csv"
    )
    powerlaw = pd.read_csv(TABLES_DIR / "powerlaw_fit.csv").iloc[0]
    return locals()




**פלט ומשמעות.** בלוק ``_result_context`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 83-92: `_top_name`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [28]:
def _top_name(centrality: pd.DataFrame, metric_prefix: str) -> str:
    row = centrality[
        centrality["metric"].str.startswith(metric_prefix)
    ].sort_values("rank").iloc[0]
    return (
        f"{row['developer_name']} (צומת {int(row['node_id'])}, "
        f"דרגה {int(row['degree']):,})"
    )




**פלט ומשמעות.** בלוק ``_top_name`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 93-105: `_overlap_value`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [29]:
def _overlap_value(
    overlap: pd.DataFrame, metric_a: str, metric_b: str
) -> float:
    mask = (
        overlap["metric_a"].str.startswith(metric_a)
        & overlap["metric_b"].str.startswith(metric_b)
    ) | (
        overlap["metric_a"].str.startswith(metric_b)
        & overlap["metric_b"].str.startswith(metric_a)
    )
    return float(overlap.loc[mask, "jaccard_overlap"].iloc[0])




**פלט ומשמעות.** בלוק ``_overlap_value`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 106-538: `main`

**מה ומדוע.** פונקציית התזמור של הקובץ. היא מוגדרת כאן אך אינה מופעלת אוטומטית; תא הרצה מפורש בהמשך מפעיל אותה ושומר פלטים.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [30]:
def main() -> None:
    context = _result_context()
    key = context["key"]
    centrality = context["centrality"]
    overlap = context["overlap"]
    community_metrics = context["community_metrics"]
    communities = context["communities"]
    classification = context["classification"]
    threshold = context["threshold"]
    ablation = context["ablation"]
    embeddings = context["embeddings"]
    reconstruction = context["reconstruction"]
    reconstruction_groups = context["reconstruction_groups"]
    reconstruction_gaps = context["reconstruction_gaps"]
    null_homophily = context["null_homophily"]
    null_modularity = context["null_modularity"]
    networks = context["networks"]
    powerlaw = context["powerlaw"]

    degree_top = _top_name(centrality, "Degree Centrality")
    pagerank_top = _top_name(centrality, "PageRank")
    betweenness_top = _top_name(centrality, "Betweenness")
    degree_pr_overlap = _overlap_value(
        overlap, "Degree Centrality", "PageRank"
    )
    degree_bet_overlap = _overlap_value(
        overlap, "Degree Centrality", "Betweenness"
    )

    top_communities = communities.head(3).copy()
    top_communities["summary"] = top_communities.apply(
        lambda row: (
            f"קהילה {int(row['community_id'])}: {int(row['size']):,} צמתים, "
            f"{100 * row['ml_ratio']:.1f}% ML"
        ),
        axis=1,
    )
    community_lines = "\n".join(
        f"- {text}" for text in top_communities["summary"]
    )

    classification_display = classification[
        ["model", "macro_f1_mean", "macro_f1_std", "roc_auc_mean", "roc_auc_std"]
    ].copy()
    classification_display.columns = [
        "מודל",
        "Macro F1 ממוצע",
        "Macro F1 סטיית תקן",
        "ROC-AUC ממוצע",
        "ROC-AUC סטיית תקן",
    ]
    for column in classification_display.columns[1:]:
        classification_display[column] = classification_display[column].map(
            lambda value: _f(value)
        )

    ablation_display = ablation[
        ["model", "macro_f1_mean", "roc_auc_mean", "roc_auc_std"]
    ].copy()
    ablation_display.columns = [
        "מקור מידע",
        "Macro F1",
        "ROC-AUC",
        "סטיית תקן AUC",
    ]
    for column in ablation_display.columns[1:]:
        ablation_display[column] = ablation_display[column].map(
            lambda value: _f(value)
        )

    embedding_display = embeddings[
        ["model", "macro_f1_mean", "roc_auc_mean", "roc_auc_std"]
    ].copy()
    embedding_display.columns = [
        "מודל",
        "Macro F1",
        "ROC-AUC",
        "סטיית תקן AUC",
    ]
    for column in embedding_display.columns[1:]:
        embedding_display[column] = embedding_display[column].map(
            lambda value: _f(value)
        )

    best_heuristic = key["best_reconstruction_heuristic"]
    best_groups = reconstruction_groups[
        reconstruction_groups["heuristic"] == best_heuristic
    ].set_index("edge_group")
    best_gap = reconstruction_gaps[
        reconstruction_gaps["heuristic"] == best_heuristic
    ].iloc[0]
    same_auc = float(best_groups.loc["same-field", "roc_auc_mean"])
    cross_auc = float(best_groups.loc["cross-field", "roc_auc_mean"])
    same_ap = float(
        best_groups.loc["same-field", "average_precision_mean"]
    )
    cross_ap = float(
        best_groups.loc["cross-field", "average_precision_mean"]
    )
    relationship = "גבוהה" if same_auc > cross_auc else "נמוכה"

    label_null = null_homophily.set_index("null_model")
    network_rows = networks.set_index("network")
    observed_network = network_rows.loc["GitHub observed"]
    er_network = network_rows.loc["Erdos-Renyi"]
    ba_name = next(name for name in network_rows.index if name.startswith("Barabasi"))
    ws_name = next(name for name in network_rows.index if name.startswith("Watts"))
    config_network = network_rows.loc[
        "Degree-preserving configuration null"
    ]

    nb = nbf.v4.new_notebook()
    nb.metadata["kernelspec"] = {
        "display_name": "Python 3",
        "language": "python",
        "name": "python3",
    }
    nb.metadata["language_info"] = {"name": "python", "version": "3"}
    nb.metadata["final_project_expected_results"] = key

    cells = []
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""# תחום התמחות של מפתחי GitHub מתוך רשת מעקב הדדית

**פרויקט גמר — אלגוריתמים ברשתות**

נועה ירין לוי · ג'ימי חלאוי · אופיר אטיאס<br>
מרצה: ד"ר אבנר פריאל

העבודה בוחנת רשת לא־מכוונת של {key['network_nodes']:,} מפתחים ו־{key['network_edges']:,} קשרי מעקב הדדיים. הגרסה הסופית מבוססת על שיטות הקורס בלבד; מאפייני תוכן משמשים נקודת ייחוס מבוקרת ולא תחליף לניתוח הרשת."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 1. מוטיבציה ושאלות מחקר

קשרי מעקב ב־GitHub עשויים לשאת מידע תיאורי על תחומי עניין מקצועיים. מטרת הניתוח אינה לטעון לסיבתיות, אלא למדוד אילו מאפיינים מבניים קשורים לתוויות Web ו־Machine Learning וכמה מידע שימושי הם מכילים.

1. **RQ1 — סיווג צמתים:** באיזו מידה שכנות, מרכזיות, קהילות וייצוגים נלמדים מאפשרים לסווג את תחום המפתח ללא שימוש בתוויות הבדיקה?
2. **RQ2 — רשת מול תוכן:** מהו הערך התיאורי השולי של מאפייני הרשת מול מאפייני התוכן, ומה מוסיף שילובם?
3. **RQ3 — שחזור קשתות:** האם ניתן לשחזר קשתות מעקב שהוסתרו אקראית באמצעות היוריסטיקות מבניות וייצוגים נלמדים, והאם קשתות תוך־תחומיות קלות יותר לשחזור מקשתות בין־תחומיות?
4. **RQ4 — מודלי אפס:** האם ההומופיליה והמודולריות הנצפות חריגות ביחס לערבוב תוויות ולגרפים משמרי־דרגות, ואילו מאפייני רשת משוחזרים על ידי מודלי ER, BA, Small-World ו־Configuration?

RQ3 עוסק בשחזור קשתות שהוסתרו מתוך תמונת מצב סטטית — לא בתחזית זמן ולא במנגנון היווצרות קשרים."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 2. סביבת הרצה ושחזוריות

כל האקראיות נשלטת באמצעות seeds קבועים: 42, 7, 21, 84 ו־123. אימון Word2Vec מתבצע על worker יחיד. להרצה ביט־ל־ביט מומלץ להגדיר `PYTHONHASHSEED=0` לפני פתיחת ה־kernel."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from pathlib import Path
import os, random, sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(SRC_DIR))
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"
SEEDS = (42, 7, 21, 84, 123)
random.seed(42)
np.random.seed(42)
print(f"Project: {PROJECT_DIR}")
print(f"PYTHONHASHSEED={os.environ.get('PYTHONHASHSEED', 'not set (set to 0 before launch)')}")"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 3. דאטה, הכנה ובדיקות איכות

הקלט הוא MUSAE GitHub Social Network (יוני 2019): רשימת קשתות, תווית בינארית לכל צומת ומאפייני תוכן דלילים. לפני כל ניתוח נבדקים כפילויות, לולאות עצמיות, תוויות חסרות, התאמת קבוצת הצמתים וקשירות."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from run_core_analysis import main as run_core_analysis
run_core_analysis()
display(pd.read_csv(TABLES_DIR / "data_quality_checks.csv"))
display(pd.read_csv(TABLES_DIR / "network_basic_stats.csv").round(5))
display(Image(filename=str(FIGURES_DIR / "label_distribution.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""### תיאור הרשת

הרשת כוללת {key['network_nodes']:,} צמתים ו־{key['network_edges']:,} קשתות, ללא לולאות או כפילויות ובמרכיב קשיר יחיד. שיעור הקשתות המחברות צמתים מאותו תחום הוא {_f(100 * key['homophily_rate'], 1)}%. זהו קשר תיאורי משמעותי, אך לבדו אינו מפריד בין השפעת שכיחות התוויות, התפלגות הדרגות ומבנה קהילתי."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """display(Image(filename=str(FIGURES_DIR / "degree_distribution.png")))
display(Image(filename=str(FIGURES_DIR / "edge_type_distribution.png")))
display(Image(filename=str(FIGURES_DIR / "sampled_network_by_label.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""## 4. מרכזיות: פופולריות, חשיבות רקורסיבית ותיווך

- **Degree Centrality** מדגישה קשרים ישירים; המוביל הוא {degree_top}.
- **PageRank** משקללת גם את חשיבות השכנים; המוביל הוא {pagerank_top}.
- **Betweenness Centrality** מדגישה צמתים על מסלולים רבים; בקירוב המבוסס על 100 מקורות, המוביל הוא {betweenness_top}.
- חפיפת Jaccard בין 20 המובילים ב־Degree וב־PageRank היא {_f(degree_pr_overlap)}, לעומת {_f(degree_bet_overlap)} בין Degree וב־Betweenness. לכן המדדים אינם חליפיים: הראשון והשני מדגישים מוקדי קשרים, בעוד Betweenness עשוי לחשוף מתווכים בין אזורים.

ב־HITS נשמר ציון Hub יחיד בלבד, משום שבגרף לא־מכוון אין הצדקה לשני פיצ'רים חופפים."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """centrality_top = pd.read_csv(TABLES_DIR / "centrality_top_developers.csv")
display(centrality_top.groupby("metric", group_keys=False).head(5).round(6))
display(pd.read_csv(TABLES_DIR / "centrality_top_overlap.csv").round(3))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""## 5. קהילות Louvain

Louvain מצא {int(community_metrics['number_of_communities'])} קהילות עם מודולריות {_f(community_metrics['modularity'])}. הקהילה הגדולה מכילה {int(community_metrics['largest_community_size']):,} מפתחים, והטוהר הממוצע המשוקלל בגודל הוא {_f(community_metrics['size_weighted_mean_purity'])}.

{community_lines}

הבדלי היחסים בין הקהילות תומכים בכך שהמבנה הקהילתי קשור לתחום, אך קהילות מעורבות מונעות זיהוי מושלם ואין לפרש את החלוקה כזהות מקצועית קשיחה."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """display(pd.read_csv(TABLES_DIR / "community_metrics.csv").round(4))
display(pd.read_csv(TABLES_DIR / "community_summary.csv").head(12).round(3))
display(Image(filename=str(FIGURES_DIR / "community_size_and_composition.png")))
display(Image(filename=str(FIGURES_DIR / "community_size_distribution.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 6. RQ1 — סיווג צמתים ללא דליפת מידע

בכל אחת מחמש החלוקות נשמר יחס 70/30 מרובד. תוויות test אינן משמשות באימון, בחישוב מאפייני שכנים/קהילה או בכיול הסף. מודל הקהילה משתמש רק בספירות תוויות מתוך train."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from run_classification import main as run_classification
run_classification()
display(pd.read_csv(TABLES_DIR / "classification_summary.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "classification_comparison.png")))
display(pd.read_csv(TABLES_DIR / "threshold_tuning_summary.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "threshold_tradeoff.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""### תוצאות RQ1

{_table_rows(classification_display, list(classification_display.columns))}

המודל החזק בקבוצה זו הוא **{key['best_classification_model']}** עם ROC-AUC ממוצע {_f(key['best_classification_roc_auc'])}. הפער מול שכנות בלבד מצביע על ערך מצטבר של מאפייני מרכזיות וקהילה. בחירת סף נעשית על validation פנימי בלבד; היא משנה את האיזון בין Precision ל־Recall ואינה משנה את ROC-AUC."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 7. RQ2 — Ablation: רשת, תוכן ושילוב

שלושת מקורות המידע נבחנים באותו מסווג ובאותן חלוקות. כך ההבדלים מיוחסים לייצוג הקלט ולא לפרוטוקול הערכה שונה."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from run_ablation import main as run_ablation
run_ablation()
display(pd.read_csv(TABLES_DIR / "ablation_model_results_summary.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "ablation_f1_auc_comparison.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""### תוצאות RQ2

{_table_rows(ablation_display, list(ablation_display.columns))}

מאפייני התוכן הם נקודת ייחוס חזקה, והרשת לבדה חלשה מהם; השילוב משפר את הממוצע ומצביע על מידע משלים. זו עדות חיזויית בתנאי החלוקות שנבדקו, לא הוכחה לכך שמקור מידע אחד גורם לתווית."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 8. ייצוגים נלמדים: DeepWalk ו־Node2Vec

שני הייצוגים נלמדים ללא תוויות וללא תוכן, עם אותו תקציב של 3 הילוכים לצומת, אורך 40, 64 ממדים ו־5 epochs. ב־Node2Vec נקבעו \(p=1, q=2\), הטיה מתונה לשכונות מקומיות. כל ייצוג נבדק לבדו ובשילוב הפיצ'רים הידניים."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from run_embeddings import main as run_embeddings
run_embeddings()
display(pd.read_csv(TABLES_DIR / "embedding_classification_summary.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "representation_comparison.png")))
display(Image(filename=str(FIGURES_DIR / "embedding_tsne.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""### השוואת הייצוגים

{_table_rows(embedding_display, list(embedding_display.columns))}

DeepWalk לבדו השיג ROC-AUC {_f(key['deepwalk_roc_auc'])}, ו־Node2Vec לבדו {_f(key['node2vec_roc_auc'])}. ההשוואה למאפיינים ידניים ולתוכן מראה עד כמה ההילוכים האקראיים מרכזים מידע שימושי במבנה הרשת. ביצועים דומים אינם הוכחת שקילות בין מקורות המידע; הם מתארים רק את החלוקות, הפרמטרים והמסווג שנבדקו."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 9. RQ3 — שחזור קשתות תוך שמירת קשירות

לכל seed נבנה עץ פורש, קשתותיו מוגנות, וקשתות test נדגמות רק מן הקשתות שאינן בעץ. לאחר ההסרה נבדקים במפורש `nx.is_connected(G_train)` והיעדר צמתים מבודדים. שמירת הקשירות חיונית למדדים מבוססי מסלולים, לקהילות ולהילוכים של DeepWalk/Node2Vec.

הדוגמאות השליליות הן non-edges אמיתיים מן הגרף המלא ומותאמות במספרן לכל אחת מקבוצות Web-Web, ML-ML ו־Web-ML. לכן same-field ו־cross-field נבדקות תחת כלל דגימה זהה."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from run_link_prediction import main as run_edge_reconstruction
run_edge_reconstruction()
display(pd.read_csv(TABLES_DIR / "edge_split_diagnostics.csv"))
display(pd.read_csv(TABLES_DIR / "edge_reconstruction_summary.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "edge_reconstruction_auc.png")))
display(pd.read_csv(TABLES_DIR / "edge_reconstruction_same_vs_cross_summary.csv").round(4))
display(pd.read_csv(TABLES_DIR / "edge_reconstruction_same_vs_cross_gaps.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "edge_reconstruction_same_vs_cross.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""### תוצאות RQ3

**{best_heuristic}** היה החזק ביותר בשחזור הכולל עם ROC-AUC {_f(key['best_reconstruction_roc_auc'])}. באותו מדד, AUC של same-field הוא {_f(same_auc)} ושל cross-field {_f(cross_auc)} (פער {_f(best_gap['roc_auc_same_minus_cross'])}); Average Precision הוא {_f(same_ap)} מול {_f(cross_ap)}.

לכן, עבור ההיוריסטיקה המובילה ובפרוטוקול המאוזן שנבדק, יכולת השחזור של קשתות תוך־תחומיות **{relationship}** מזו של קשתות בין־תחומיות. התוצאה אינה מוכיחה מנגנון סיבתי של יצירת קשרים ואינה ניתנת להכללה לתחזית זמן."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 10. RQ4 — מודלי אפס ומודלי רשת

נבדקים 1,000 ערבובי תוויות ו־20 גרפים משמרי־דרגות. ערכי p אמפיריים מחושבים עם תיקון finite-simulation: \((b+1)/(B+1)\), ולכן אינם יכולים להיות אפס. בנוסף מושווים מדדי הרשת למודלי ER, BA, Watts-Strogatz ולגרף קונפיגורציה פשוט משמר־דרגות."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from run_null_models import main as run_null_models
run_null_models()
display(pd.read_csv(TABLES_DIR / "null_model_homophily.csv").round(6))
display(pd.read_csv(TABLES_DIR / "null_model_modularity.csv").round(6))
display(Image(filename=str(FIGURES_DIR / "homophily_null_distribution.png")))
display(pd.read_csv(TABLES_DIR / "network_vs_random_models.csv").round(4))
display(pd.read_csv(TABLES_DIR / "powerlaw_fit.csv").round(4))
display(Image(filename=str(FIGURES_DIR / "degree_distribution_loglog.png")))"""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            f"""### תוצאות RQ4

- ההומופיליה הנצפית ({_f(key['homophily_rate'])}) גבוהה מממוצע ערבוב התוויות ({_f(label_null.loc['label_shuffle', 'null_mean'])}) ומממוצע הגרפים משמרי־הדרגות ({_f(label_null.loc['degree_preserving_rewire', 'null_mean'])}). ערכי p המתוקנים הם {_f(key['label_shuffle_empirical_p'], 6)} ו־{_f(key['degree_rewire_empirical_p'], 6)}.
- המודולריות הנצפית היא {_f(null_modularity['observed_modularity'])}, מול {_f(null_modularity['rewired_null_mean'])} בממוצע בגרפים משמרי־הדרגות; p אמפירי {_f(null_modularity['empirical_p'], 6)}.
- ההתקבצות הממוצעת ברשת היא {_f(observed_network['avg_clustering'])}, מול {_f(er_network['avg_clustering'], 4)} ב־ER, {_f(network_rows.loc[ba_name, 'avg_clustering'], 4)} ב־BA ו־{_f(network_rows.loc[ws_name, 'avg_clustering'])} ב־Small-World. מודל הקונפיגורציה משמר את הדרגות אך משחזר התקבצות של {_f(config_network['avg_clustering'])}.
- התאמת הזנב נותנת \\(\\alpha={_f(powerlaw['alpha'], 2)}\\), אך ההשוואה ללוג־נורמלית אינה מצדיקה הכרזה גורפת על רשת scale-free. המסקנה הבטוחה היא זנב כבד."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 11. דיון, מגבלות ומסקנות

**RQ1:** מבנה הרשת מספק אות חיזויי עקבי; קהילות וייצוגים נלמדים משפרים את קווי הבסיס.<br>
**RQ2:** תוכן חזק מן הפיצ'רים הידניים של הרשת, אך השילוב מוסיף מידע — חפיפה חלקית, לא שקילות.<br>
**RQ3:** שחזור קשתות אפשרי מעל רמת אקראי תוך שמירת קשירות. ההבדל same/cross תלוי בהיוריסטיקה ובפרוטוקול, ולכן מנוסח כתוצאה תיאורית.<br>
**RQ4:** ההומופיליה והמודולריות חריגות ביחס למודלי האפס שנבדקו; הבדלי דרגות מסבירים חלק ניכר אך לא את כל הפער.

### מגבלות

- הנתונים הם תמונת מצב מיוני 2019 ותווית היעד בינארית ורועשת אפשרית.
- זהו ניסוי transductive; מבנה הגרף כולו גלוי בסיווג הצמתים.
- Betweenness מחושב בקירוב בגלל גודל הרשת.
- פרמטרי ההטמעה נקבעו מראש ולא בוצע חיפוש נרחב.
- שחזור קשתות שהוסתרו אינו תחזית זמן; לא נבדקו כיווניות, עוצמת קשר או סיבתיות.
- מודלי האפס והמודלים הגנרטיביים הם השוואות ממוקדות, לא תיאור מלא של כל תהליכי הרשת.

בסך הכול נבדקו **10 מודלי סיווג לוגיים**: ארבעה מודלי בסיס/פיצ'רים ידניים, תוכן והיברידי, וארבעה מודלים המבוססים על DeepWalk/Node2Vec לבדם ובשילוב ידני."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 12. מקורות

1. Rozemberczki, B., Allen, C., & Sarkar, R. (2019). *Multi-scale Attributed Node Embedding (MUSAE)*.
2. Blondel, V. D., et al. (2008). *Fast unfolding of communities in large networks*.
3. Perozzi, B., Al-Rfou, R., & Skiena, S. (2014). *DeepWalk*.
4. Grover, A., & Leskovec, J. (2016). *node2vec: Scalable Feature Learning for Networks*.
5. Liben-Nowell, D., & Kleinberg, J. (2003). *The Link Prediction Problem for Social Networks*.
6. Clauset, A., Shalizi, C. R., & Newman, M. E. J. (2009). *Power-law distributions in empirical data*.
7. Page, L., et al. (1999). *The PageRank Citation Ranking*.
8. מצגות ותרגולי הקורס בנושאי מרכזיות, קהילות, מודלי רשת, שחזור קשתות והטמעות גרף."""
        )
    )
    cells.append(
        nbf.v4.new_markdown_cell(
            """## 13. Validation סופי

התא הבא נכשל אם חסר פלט, CSV ריק או לא־סופי, מדד מחוץ לטווח, מודל לא מסונכרן, ערך p לא תקין, גרף אימון מנותק, צומת מבודד, חוסר התאמה בין ערכי המחברת לקובצי ה־CSV, או דוח החורג משמונה עמודים."""
        )
    )
    cells.append(
        nbf.v4.new_code_cell(
            """from final_validation import run_all_validations
run_all_validations(PROJECT_DIR, verify_notebook_execution=False)"""
        )
    )

    nb["cells"] = cells
    NOTEBOOK_PATH.parent.mkdir(parents=True, exist_ok=True)
    nbf.write(nb, NOTEBOOK_PATH)
    print(f"Built notebook: {NOTEBOOK_PATH}")




**פלט ומשמעות.** בלוק ``main`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_notebook.py, שורות 539-540: `__main__ guard`

**מה ומדוע.** זהו שער ההרצה המקורי של קובץ CLI. במחברת הוא נשמר כלשונו, אך אינו מופעל כי `__name__` הוגדר כשם מחברת; ההרצה נעשית בתא מפורש ומתועד.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** אין פלט, משום ששער ה-CLI מושבת בכוונה בסביבת המחברת.

In [31]:
if __name__ == "__main__":
    main()


**פלט ומשמעות.** בלוק ``__main__ guard`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

## קוד המקור: `src/build_readme.py`

יצירת README מתוך טבלאות התוצאות, כדי למנוע מספרים מיושנים בתיעוד.

לפני טעינת הקטעים נגדיר עבורם `__file__` וירטואלי התואם למיקומם המקורי. הדבר משמר את לוגיקת חישוב `PROJECT_DIR` בלי לשנות שורה בקוד המקור. SHA-256 של המקור: `2dd362286366efe0e20e83f026c1b79e299aadc3dbddc0aadb4dfa63524b8322`.

התא הבא מכין את הקשר הנתיבים עבור קובץ המקור. הוא אינו משנה את הקובץ; הפלט הצפוי הוא נתיב המקור שממנו הועתק הקוד.

In [32]:
__file__ = str(ROOT / 'src/build_readme.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\build_readme.py


**פלט ומשמעות.** הנתיב המודפס מאפשר לבקר שהקבועים בקוד יחושבו ביחס לריפוזיטורי הנוכחי בלבד.

### build_readme.py, שורות 1-15: ייבואים, קבועים והגדרות מודול

**מה ומדוע.** הבלוק מרכז את התלויות, הקבועים והנחות ברירת המחדל של המודול; בלעדיו הפונקציות הבאות אינן עצמאיות.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [33]:
# -*- coding: utf-8 -*-
"""Generate README.md from the current result tables."""

from pathlib import Path
import sys

import pandas as pd

SRC_DIR = Path(__file__).resolve().parent
PROJECT_DIR = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))

from final_validation import collect_key_results




**פלט ומשמעות.** בלוק `ייבואים, קבועים והגדרות מודול` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_readme.py, שורות 16-19: `_f`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [34]:
def _f(value: float, digits: int = 3) -> str:
    return f"{float(value):.{digits}f}"




**פלט ומשמעות.** בלוק ``_f`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_readme.py, שורות 20-256: `main`

**מה ומדוע.** פונקציית התזמור של הקובץ. היא מוגדרת כאן אך אינה מופעלת אוטומטית; תא הרצה מפורש בהמשך מפעיל אותה ושומר פלטים.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [35]:
def main() -> None:
    key = collect_key_results(PROJECT_DIR)
    tables = PROJECT_DIR / "outputs" / "tables"
    ablation = pd.read_csv(
        tables / "ablation_model_results_summary.csv"
    ).set_index("model")
    groups = pd.read_csv(
        tables / "edge_reconstruction_same_vs_cross_summary.csv"
    )
    best_groups = groups[
        groups["heuristic"] == key["best_reconstruction_heuristic"]
    ].set_index("edge_group")
    modularity = pd.read_csv(
        tables / "null_model_modularity.csv"
    ).iloc[0]
    content_auc = ablation.loc[
        "Logistic Regression - Content Features Only", "roc_auc_mean"
    ]
    network_auc = ablation.loc[
        "Logistic Regression - Network Only (Graph + Community)",
        "roc_auc_mean",
    ]
    hybrid_auc = ablation.loc[
        "Logistic Regression - Hybrid (Network + Content)",
        "roc_auc_mean",
    ]

    readme = f"""# תחום התמחות של מפתחי GitHub מתוך רשת מעקב הדדית

פרויקט גמר בקורס **אלגוריתמים ברשתות**. הפרויקט מנתח רשת לא־מכוונת של
{key['network_nodes']:,} מפתחי GitHub ו־{key['network_edges']:,} קשרי מעקב
הדדיים, ובוחן כמה מידע על התחום המקצועי (Web מול Machine Learning) נמצא
בשכנות, במרכזיות, בקהילות ובהטמעות גרף.

הגרסה המרכזית משתמשת בשיטות הקורס בלבד. תיקיית `extensions/` שומרת ניסויים
אופציונליים שאינם חלק מהצנרת, מהתוצאות או מטענות ההגשה.

## שאלות המחקר

1. **RQ1 — סיווג צמתים:** באיזו מידה שכנות, מרכזיות, קהילות וייצוגים נלמדים
   מאפשרים לסווג את תחום המפתח ללא שימוש בתוויות הבדיקה?
2. **RQ2 — רשת מול תוכן:** מהו הערך השולי של מבנה הרשת מול מאפייני התוכן,
   ומה מוסיף שילובם?
3. **RQ3 — שחזור קשתות:** האם ניתן לשחזר קשתות מעקב שהוסתרו אקראית באמצעות
   היוריסטיקות מבניות וייצוגים נלמדים, והאם קשתות תוך־תחומיות קלות יותר
   לשחזור מקשתות בין־תחומיות?
4. **RQ4 — מודלי אפס ורשת:** האם ההומופיליה והמודולריות הנצפות חריגות ביחס
   לערבוב תוויות ולגרפים משמרי־דרגות, ואילו מאפיינים משחזרים ER, BA,
   Small-World ו־Configuration?

RQ3 עוסק **בשחזור קשתות שהוסתרו מתמונת מצב סטטית**, לא בתחזית זמן ולא
בהסקת מנגנון סיבתי.

## הדאטה

הנתונים הם MUSAE GitHub Social Network (יוני 2019):

- `data/musae_git_edges.csv` — {key['network_edges']:,} קשתות ללא לולאות או
  כפילויות.
- `data/musae_git_target.csv` — תווית בינארית מלאה לכל {key['network_nodes']:,}
  הצמתים.
- `data/musae_git_features.json` — מאפייני תוכן דלילים המשמשים רק בניסוי
  ההשוואה רשת/תוכן.
- הרשת היא רכיב קשיר יחיד; שיעור הקשתות התוך־תחומיות הוא
  {_f(100 * key['homophily_rate'], 1)}%.

## אלגוריתמים בגרסה המרכזית

- ניתוח מבני: Degree Centrality, Clustering Coefficient, PageRank,
  Betweenness Centrality (בקירוב), HITS Hub יחיד, Louvain ומודולריות.
- מודלי רשת: Erdős–Rényi, Barabási–Albert, Watts–Strogatz Small-World,
  Configuration/degree-preserving null ו־Power Law.
- סיווג: Majority, Neighbor Majority ורגרסיה לוגיסטית על פיצ'רים ידניים,
  קהילה, תוכן ושילובים.
- ייצוגים נלמדים: DeepWalk ו־Node2Vec, לבדם ובשילוב פיצ'רים ידניים.
- שחזור קשתות: Common Neighbors, Jaccard, Adamic-Adar, Preferential
  Attachment, Same Community, DeepWalk ו־Node2Vec.
- מודלי אפס: 1,000 ערבובי תוויות ו־20 גרפים משמרי־דרגות, עם
  `(extreme_count + 1) / (num_simulations + 1)`.

בסך הכול מדוּוחים **10 מודלי סיווג לוגיים**: ארבעה קווי בסיס/מודלים ידניים,
תוכן והיברידי, וארבעה מודלי DeepWalk/Node2Vec לבדם ובשילוב ידני.

## תוצאות מרכזיות מעודכנות

- הסיווג הידני החזק ביותר הוא **{key['best_classification_model']}** עם
  ROC-AUC ממוצע {_f(key['best_classification_roc_auc'])}.
- רשת ידנית מול תוכן מול היברידי: {_f(network_auc)} מול {_f(content_auc)}
  מול {_f(hybrid_auc)} ROC-AUC.
- DeepWalk לבדו: {_f(key['deepwalk_roc_auc'])}; Node2Vec לבדו:
  {_f(key['node2vec_roc_auc'])} ROC-AUC. דמיון בביצועים אינו הוכחת שקילות
  בין מקורות מידע.
- **{key['best_reconstruction_heuristic']}** מוביל בשחזור הקשתות עם
  ROC-AUC {_f(key['best_reconstruction_roc_auc'])}.
- עבור שיטה זו, same-field מול cross-field:
  {_f(best_groups.loc['same-field', 'roc_auc_mean'])} מול
  {_f(best_groups.loc['cross-field', 'roc_auc_mean'])} ROC-AUC,
  ו־{_f(best_groups.loc['same-field', 'average_precision_mean'])} מול
  {_f(best_groups.loc['cross-field', 'average_precision_mean'])} Average
  Precision.
- ערכי p האמפיריים הם
  {_f(key['label_shuffle_empirical_p'], 6)} (ערבוב תוויות),
  {_f(key['degree_rewire_empirical_p'], 6)} (גרף משמר־דרגות) ו־
  {_f(key['modularity_empirical_p'], 6)} (מודולריות); אף אחד מהם אינו אפס.
- המודולריות הנצפית היא {_f(modularity['observed_modularity'])}, מול
  {_f(modularity['rewired_null_mean'])} בממוצע בגרפים משמרי־דרגות.

## תכנון הניסויים

- כל תוצאות הסיווג מדווחות כממוצע וסטיית תקן על seeds
  `42, 7, 21, 84, 123`.
- מאפיינים המבוססים על תוויות משתמשים בתוויות train בלבד.
- בחירת סף נעשית על validation פנימי, לא על test.
- DeepWalk ו־Node2Vec נלמדים ללא תוויות.
- בשחזור קשתות מוגן עץ פורש; כל קשתות test מגיעות מן הקשתות שאינן בעץ.
  כל גרף אימון נבדק עם `assert nx.is_connected(G_train)` וללא צמתים
  מבודדים.
- non-edges נדגמים באותו מספר בתוך Web-Web, ML-ML ו־Web-ML, כך שהשוואת
  same-field/cross-field הוגנת.

## מבנה הריפוזיטורי

```text
data/                 נתוני MUSAE המקוריים
src/                  קוד הצנרת, builders ו-validation
extensions/           הרחבות אופציונליות מחוץ להגשה המרכזית
notebooks/            המחברת הסופית והמורצת
outputs/tables/       כל טבלאות ה-CSV העדכניות
outputs/figures/      כל הגרפים העדכניים ברזולוציה גבוהה
outputs/embeddings/   קובצי DeepWalk ו-Node2Vec
report/report_source.docx   מקור העריכה של הדוח
report/final_report.pdf     הדוח הסופי (עד 8 עמודים)
tests/                בדיקות יחידה לפרוטוקולים הקריטיים
```

## התקנה והרצה — Windows PowerShell

```powershell
cd C:\\path\\to\\github-social-network
py -3.11 -m venv .venv
.\\.venv\\Scripts\\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
$env:PYTHONHASHSEED="0"

# מוחק רק פלטים ישנים תחת outputs/ ובונה הכול מחדש
python src/run_all.py

# ה-PDF וה-DOCX נבנו כבר ב-run_all; לבנייה ממוקדת של ה-PDF:
python src/build_report_pdf.py

# הרצת המחברת כולה מתוך הריפוזיטורי
python -m nbconvert --to notebook --execute --inplace `
  --ExecutePreprocessor.timeout=-1 `
  notebooks/GitHub_Developer_Field_Prediction_Network_Only.ipynb

python src/final_validation.py
```

## התקנה והרצה — Linux/macOS

```bash
cd /path/to/github-social-network
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
export PYTHONHASHSEED=0

python src/run_all.py

# ה-PDF וה-DOCX נבנו כבר ב-run_all; לבנייה ממוקדת של ה-PDF:
python src/build_report_pdf.py

python -m nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 \
  notebooks/GitHub_Developer_Field_Prediction_Network_Only.ipynb

python src/final_validation.py
```

## הרצה ממוקדת

```bash
python src/run_core_analysis.py
python src/run_classification.py
python src/run_ablation.py
python src/run_embeddings.py
python src/run_link_prediction.py
python src/run_null_models.py
python src/build_notebook.py
python src/build_report.py
python src/build_report_pdf.py
```

`python src/run_all.py` הוא המסלול הסמכותי: הוא מנקה את `outputs/`, מריץ את
השלבים בסדר הנכון ובונה מחדש את המחברת, ה־README, מקור הדוח וה־PDF הסופי.

## Validation סופי

`python src/final_validation.py` נכשל אם:

- חסר פלט או CSV ריק/לא־קריא;
- יש NaN/Inf או מדד מחוץ לטווח;
- מספרי הצמתים/קשתות אינם תואמים;
- יש לולאות, כפילויות או תוויות חסרות;
- אחד מחמשת גרפי האימון מנותק או מכיל מבודדים;
- positive/negative אינם מותאמים לפי סוג קשת;
- ערך p אמפירי הוא אפס או גדול מאחד;
- מודל חסר/עודף או פלט ישן נשאר;
- ערכי המחברת אינם תואמים ל־CSV;
- הדוח חסר מקור עריכה או חורג משמונה עמודים.

רק לאחר כל הבדיקות מודפסת ההודעה:

```text
ALL FINAL PROJECT VALIDATIONS PASSED
```

## מגבלות

- הנתונים נאספו ביוני 2019 ותווית היעד בינארית.
- הסיווג transductive ומניח שמבנה הגרף ידוע.
- Betweenness מקורב; תקציב ההטמעות קבוע ולא בוצע חיפוש פרמטרים נרחב.
- שחזור קשתות שהוסתרו אינו תחזית זמן ואינו הוכחה סיבתית.
- מודלי האפס והמודלים הגנרטיביים מבודדים מאפיינים מסוימים ואינם תיאור מלא
  של כל תהליכי הרשת.

## מקורות עיקריים

MUSAE; Louvain; PageRank; DeepWalk; Node2Vec; Link Prediction; Power-law
distributions; ומצגות/תרגולי הקורס בנושאי מרכזיות, קהילות ומודלי רשת.
"""
    (PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
    print(f"Built README: {PROJECT_DIR / 'README.md'}")




**פלט ומשמעות.** בלוק ``main`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_readme.py, שורות 257-258: `__main__ guard`

**מה ומדוע.** זהו שער ההרצה המקורי של קובץ CLI. במחברת הוא נשמר כלשונו, אך אינו מופעל כי `__name__` הוגדר כשם מחברת; ההרצה נעשית בתא מפורש ומתועד.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** אין פלט, משום ששער ה-CLI מושבת בכוונה בסביבת המחברת.

In [36]:
if __name__ == "__main__":
    main()


**פלט ומשמעות.** בלוק ``__main__ guard`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

## קוד המקור: `src/build_report.py`

יצירת מקור DOCX עריך לדוח האקדמי.

לפני טעינת הקטעים נגדיר עבורם `__file__` וירטואלי התואם למיקומם המקורי. הדבר משמר את לוגיקת חישוב `PROJECT_DIR` בלי לשנות שורה בקוד המקור. SHA-256 של המקור: `a04fc03840c695b12ae4047ff75c6fdff37f3e4fe7a1c6ce6fdad2a01816d7f3`.

התא הבא מכין את הקשר הנתיבים עבור קובץ המקור. הוא אינו משנה את הקובץ; הפלט הצפוי הוא נתיב המקור שממנו הועתק הקוד.

In [37]:
__file__ = str(ROOT / 'src/build_report.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\build_report.py


**פלט ומשמעות.** הנתיב המודפס מאפשר לבקר שהקבועים בקוד יחושבו ביחס לריפוזיטורי הנוכחי בלבד.

### build_report.py, שורות 1-42: ייבואים, קבועים והגדרות מודול

**מה ומדוע.** הבלוק מרכז את התלויות, הקבועים והנחות ברירת המחדל של המודול; בלעדיו הפונקציות הבאות אינן עצמאיות.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [38]:
# -*- coding: utf-8 -*-
"""Create the editable eight-page academic report from fresh output tables."""

from pathlib import Path
import sys

import pandas as pd
from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_ALIGN_VERTICAL, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

SRC_DIR = Path(__file__).resolve().parent
PROJECT_DIR = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))

from final_validation import collect_key_results


REPORT_DIR = PROJECT_DIR / "report"
REPORT_PATH = REPORT_DIR / "report_source.docx"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"

# Compact-reference-guide base with a named academic page-limit override.
PAGE_WIDTH_DXA = 12_240
PAGE_HEIGHT_DXA = 15_840
MARGIN_DXA = 900
CONTENT_WIDTH_DXA = PAGE_WIDTH_DXA - 2 * MARGIN_DXA
TABLE_INDENT_DXA = 100
CELL_MARGIN_DXA = {"top": 55, "bottom": 55, "start": 100, "end": 100}
INK = "172B4D"
ACCENT = "315C8D"
ACCENT_2 = "D57837"
MUTED = "5F6B7A"
LIGHT_FILL = "E8EEF5"
PALE_FILL = "F4F6F9"




**פלט ומשמעות.** בלוק `ייבואים, קבועים והגדרות מודול` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 43-46: `_rgb`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [39]:
def _rgb(hex_color: str) -> RGBColor:
    return RGBColor.from_string(hex_color)




**פלט ומשמעות.** בלוק ``_rgb`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 47-64: `_set_run_font`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [40]:
def _set_run_font(
    run,
    *,
    size: float,
    bold: bool = False,
    color: str = "000000",
    italic: bool = False,
) -> None:
    run.font.name = "Calibri"
    run._element.get_or_add_rPr().rFonts.set(qn("w:ascii"), "Calibri")
    run._element.get_or_add_rPr().rFonts.set(qn("w:hAnsi"), "Calibri")
    run._element.get_or_add_rPr().rFonts.set(qn("w:cs"), "Arial")
    run.font.size = Pt(size)
    run.font.bold = bold
    run.font.italic = italic
    run.font.color.rgb = _rgb(color)




**פלט ומשמעות.** בלוק ``_set_run_font`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 65-73: `_set_rtl`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [41]:
def _set_rtl(paragraph, alignment=WD_ALIGN_PARAGRAPH.RIGHT) -> None:
    paragraph.alignment = alignment
    p_pr = paragraph._p.get_or_add_pPr()
    bidi = p_pr.find(qn("w:bidi"))
    if bidi is None:
        bidi = OxmlElement("w:bidi")
        p_pr.append(bidi)




**פלט ומשמעות.** בלוק ``_set_rtl`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 74-103: `_configure_styles`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [42]:
def _configure_styles(document: Document) -> None:
    normal = document.styles["Normal"]
    normal.font.name = "Calibri"
    normal._element.rPr.rFonts.set(qn("w:ascii"), "Calibri")
    normal._element.rPr.rFonts.set(qn("w:hAnsi"), "Calibri")
    normal._element.rPr.rFonts.set(qn("w:cs"), "Arial")
    normal.font.size = Pt(9.3)
    normal.paragraph_format.space_before = Pt(0)
    normal.paragraph_format.space_after = Pt(3)
    normal.paragraph_format.line_spacing = 1.03

    heading_tokens = {
        "Heading 1": (13.5, 7, 3, ACCENT),
        "Heading 2": (11.0, 5, 2, ACCENT),
        "Heading 3": (10.0, 4, 1.5, INK),
    }
    for style_name, (size, before, after, color) in heading_tokens.items():
        style = document.styles[style_name]
        style.font.name = "Calibri"
        style._element.rPr.rFonts.set(qn("w:ascii"), "Calibri")
        style._element.rPr.rFonts.set(qn("w:hAnsi"), "Calibri")
        style._element.rPr.rFonts.set(qn("w:cs"), "Arial")
        style.font.size = Pt(size)
        style.font.bold = True
        style.font.color.rgb = _rgb(color)
        style.paragraph_format.space_before = Pt(before)
        style.paragraph_format.space_after = Pt(after)
        style.paragraph_format.keep_with_next = True




**פלט ומשמעות.** בלוק ``_configure_styles`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 104-115: `_configure_section`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [43]:
def _configure_section(section) -> None:
    section.page_width = Inches(8.5)
    section.page_height = Inches(11)
    margin = Inches(MARGIN_DXA / 1440)
    section.top_margin = margin
    section.bottom_margin = margin
    section.left_margin = margin
    section.right_margin = margin
    section.header_distance = Inches(0.28)
    section.footer_distance = Inches(0.28)




**פלט ומשמעות.** בלוק ``_configure_section`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 116-132: `_add_field`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [44]:
def _add_field(paragraph, instruction: str) -> None:
    run = paragraph.add_run()
    begin = OxmlElement("w:fldChar")
    begin.set(qn("w:fldCharType"), "begin")
    instr = OxmlElement("w:instrText")
    instr.set(qn("xml:space"), "preserve")
    instr.text = instruction
    separate = OxmlElement("w:fldChar")
    separate.set(qn("w:fldCharType"), "separate")
    text = OxmlElement("w:t")
    text.text = "1"
    end = OxmlElement("w:fldChar")
    end.set(qn("w:fldCharType"), "end")
    run._r.extend([begin, instr, separate, text, end])
    _set_run_font(run, size=8, color=MUTED)




**פלט ומשמעות.** בלוק ``_add_field`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 133-151: `_set_header_footer`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [45]:
def _set_header_footer(section) -> None:
    header = section.header
    paragraph = header.paragraphs[0]
    _set_rtl(paragraph, WD_ALIGN_PARAGRAPH.RIGHT)
    paragraph.paragraph_format.space_after = Pt(0)
    run = paragraph.add_run("תחום התמחות ברשת GitHub | פרויקט גמר")
    _set_run_font(run, size=7.8, color=MUTED)

    footer = section.footer
    paragraph = footer.paragraphs[0]
    _set_rtl(paragraph, WD_ALIGN_PARAGRAPH.CENTER)
    paragraph.paragraph_format.space_before = Pt(0)
    run = paragraph.add_run("עמוד ")
    _set_run_font(run, size=8, color=MUTED)
    _add_field(paragraph, "PAGE")
    run = paragraph.add_run(" מתוך 8")
    _set_run_font(run, size=8, color=MUTED)




**פלט ומשמעות.** בלוק ``_set_header_footer`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 152-176: `_add_paragraph`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [46]:
def _add_paragraph(
    document: Document,
    text: str,
    *,
    bold_lead: str | None = None,
    size: float = 9.3,
    color: str = "000000",
    after: float = 3,
    align=WD_ALIGN_PARAGRAPH.RIGHT,
    keep_with_next: bool = False,
) -> None:
    paragraph = document.add_paragraph()
    _set_rtl(paragraph, align)
    paragraph.paragraph_format.space_after = Pt(after)
    paragraph.paragraph_format.keep_with_next = keep_with_next
    if bold_lead and text.startswith(bold_lead):
        lead = paragraph.add_run(bold_lead)
        _set_run_font(lead, size=size, bold=True, color=color)
        tail = paragraph.add_run(text[len(bold_lead) :])
        _set_run_font(tail, size=size, color=color)
    else:
        run = paragraph.add_run(text)
        _set_run_font(run, size=size, color=color)




**פלט ומשמעות.** בלוק ``_add_paragraph`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 177-185: `_add_heading`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [47]:
def _add_heading(document: Document, text: str, level: int = 1) -> None:
    paragraph = document.add_paragraph(style=f"Heading {level}")
    _set_rtl(paragraph)
    run = paragraph.add_run(text)
    size = {1: 13.5, 2: 11.0, 3: 10.0}[level]
    color = ACCENT if level < 3 else INK
    _set_run_font(run, size=size, bold=True, color=color)




**פלט ומשמעות.** בלוק ``_add_heading`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 186-196: `_add_bullet`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי. תיעוד המקור מסביר: Add a real Word-numbering bullet with compact report spacing.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [48]:
def _add_bullet(document: Document, text: str) -> None:
    """Add a real Word-numbering bullet with compact report spacing."""
    paragraph = document.add_paragraph(style="List Bullet")
    _set_rtl(paragraph)
    paragraph.paragraph_format.left_indent = Inches(0.22)
    paragraph.paragraph_format.first_line_indent = Inches(-0.14)
    paragraph.paragraph_format.space_after = Pt(1.2)
    run = paragraph.add_run(text)
    _set_run_font(run, size=8.8)




**פלט ומשמעות.** בלוק ``_add_bullet`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 197-205: `_shade_cell`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [49]:
def _shade_cell(cell, color: str) -> None:
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = tc_pr.find(qn("w:shd"))
    if shd is None:
        shd = OxmlElement("w:shd")
        tc_pr.append(shd)
    shd.set(qn("w:fill"), color)




**פלט ומשמעות.** בלוק ``_shade_cell`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 206-221: `_set_cell_margins`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [50]:
def _set_cell_margins(cell) -> None:
    tc_pr = cell._tc.get_or_add_tcPr()
    tc_mar = tc_pr.find(qn("w:tcMar"))
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for edge, value in CELL_MARGIN_DXA.items():
        tag = qn(f"w:{edge}")
        node = tc_mar.find(tag)
        if node is None:
            node = OxmlElement(f"w:{edge}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(value))
        node.set(qn("w:type"), "dxa")




**פלט ומשמעות.** בלוק ``_set_cell_margins`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 222-228: `_set_repeat_header`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [51]:
def _set_repeat_header(row) -> None:
    tr_pr = row._tr.get_or_add_trPr()
    tbl_header = OxmlElement("w:tblHeader")
    tbl_header.set(qn("w:val"), "true")
    tr_pr.append(tbl_header)




**פלט ומשמעות.** בלוק ``_set_repeat_header`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 229-234: `_prevent_row_split`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [52]:
def _prevent_row_split(row) -> None:
    tr_pr = row._tr.get_or_add_trPr()
    cant_split = OxmlElement("w:cantSplit")
    tr_pr.append(cant_split)




**פלט ומשמעות.** בלוק ``_prevent_row_split`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 235-276: `_set_table_geometry`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [53]:
def _set_table_geometry(table, widths_dxa: list[int]) -> None:
    if sum(widths_dxa) != CONTENT_WIDTH_DXA:
        raise ValueError("Table widths must equal the configured content width.")
    table.autofit = False
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    tbl_pr = table._tbl.tblPr
    tbl_w = tbl_pr.find(qn("w:tblW"))
    if tbl_w is None:
        tbl_w = OxmlElement("w:tblW")
        tbl_pr.append(tbl_w)
    tbl_w.set(qn("w:w"), str(CONTENT_WIDTH_DXA))
    tbl_w.set(qn("w:type"), "dxa")
    tbl_ind = tbl_pr.find(qn("w:tblInd"))
    if tbl_ind is None:
        tbl_ind = OxmlElement("w:tblInd")
        tbl_pr.append(tbl_ind)
    tbl_ind.set(qn("w:w"), str(TABLE_INDENT_DXA))
    tbl_ind.set(qn("w:type"), "dxa")

    grid = table._tbl.tblGrid
    for child in list(grid):
        grid.remove(child)
    for width in widths_dxa:
        grid_col = OxmlElement("w:gridCol")
        grid_col.set(qn("w:w"), str(width))
        grid.append(grid_col)

    for row in table.rows:
        _prevent_row_split(row)
        for cell, width in zip(row.cells, widths_dxa):
            cell.width = Inches(width / 1440)
            cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            _set_cell_margins(cell)
            tc_pr = cell._tc.get_or_add_tcPr()
            tc_w = tc_pr.find(qn("w:tcW"))
            if tc_w is None:
                tc_w = OxmlElement("w:tcW")
                tc_pr.append(tc_w)
            tc_w.set(qn("w:w"), str(width))
            tc_w.set(qn("w:type"), "dxa")




**פלט ומשמעות.** בלוק ``_set_table_geometry`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 277-294: `_set_cell_text`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [54]:
def _set_cell_text(
    cell,
    text: str,
    *,
    bold: bool = False,
    size: float = 8.2,
    color: str = "000000",
    alignment=WD_ALIGN_PARAGRAPH.RIGHT,
) -> None:
    cell.text = ""
    paragraph = cell.paragraphs[0]
    _set_rtl(paragraph, alignment)
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    run = paragraph.add_run(str(text))
    _set_run_font(run, size=size, bold=bold, color=color)




**פלט ומשמעות.** בלוק ``_set_cell_text`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 295-317: `_add_table`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [55]:
def _add_table(
    document: Document,
    headers: list[str],
    rows: list[list[str]],
    widths_dxa: list[int],
    *,
    font_size: float = 8.2,
) -> None:
    table = document.add_table(rows=1, cols=len(headers))
    table.style = "Table Grid"
    for cell, header in zip(table.rows[0].cells, headers):
        _shade_cell(cell, LIGHT_FILL)
        _set_cell_text(cell, header, bold=True, size=font_size, color=INK)
    _set_repeat_header(table.rows[0])
    for row_values in rows:
        cells = table.add_row().cells
        for cell, value in zip(cells, row_values):
            _set_cell_text(cell, value, size=font_size)
    _set_table_geometry(table, widths_dxa)
    spacer = document.add_paragraph()
    spacer.paragraph_format.space_after = Pt(1)




**פלט ומשמעות.** בלוק ``_add_table`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 318-340: `_add_picture`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [56]:
def _add_picture(
    document: Document,
    filename: str,
    caption: str,
    *,
    width: float,
) -> None:
    path = FIGURES_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    paragraph.paragraph_format.keep_with_next = True
    run = paragraph.add_run()
    run.add_picture(str(path), width=Inches(width))
    caption_p = document.add_paragraph()
    _set_rtl(caption_p, WD_ALIGN_PARAGRAPH.CENTER)
    caption_p.paragraph_format.space_before = Pt(1)
    caption_p.paragraph_format.space_after = Pt(3)
    caption_run = caption_p.add_run(caption)
    _set_run_font(caption_run, size=7.8, italic=True, color=MUTED)




**פלט ומשמעות.** בלוק ``_add_picture`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 341-344: `_page_break`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [57]:
def _page_break(document: Document) -> None:
    document.add_page_break()




**פלט ומשמעות.** בלוק ``_page_break`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 345-348: `_r`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [58]:
def _r(value: float, digits: int = 3) -> str:
    return f"{float(value):.{digits}f}"




**פלט ומשמעות.** בלוק ``_r`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 349-841: `main`

**מה ומדוע.** פונקציית התזמור של הקובץ. היא מוגדרת כאן אך אינה מופעלת אוטומטית; תא הרצה מפורש בהמשך מפעיל אותה ושומר פלטים.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [59]:
def main() -> None:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    key = collect_key_results(PROJECT_DIR)
    basic = pd.read_csv(TABLES_DIR / "network_basic_stats.csv").iloc[0]
    centrality = pd.read_csv(TABLES_DIR / "centrality_top_developers.csv")
    overlap = pd.read_csv(TABLES_DIR / "centrality_top_overlap.csv")
    community_metrics = pd.read_csv(TABLES_DIR / "community_metrics.csv").iloc[0]
    communities = pd.read_csv(TABLES_DIR / "community_summary.csv")
    classification = pd.read_csv(TABLES_DIR / "classification_summary.csv")
    thresholds = pd.read_csv(TABLES_DIR / "threshold_tuning_summary.csv").set_index(
        "evaluation_type"
    )
    ablation = pd.read_csv(TABLES_DIR / "ablation_model_results_summary.csv")
    embeddings = pd.read_csv(TABLES_DIR / "embedding_classification_summary.csv")
    reconstruction = pd.read_csv(TABLES_DIR / "edge_reconstruction_summary.csv")
    reconstruction_groups = pd.read_csv(
        TABLES_DIR / "edge_reconstruction_same_vs_cross_summary.csv"
    )
    null_homophily = pd.read_csv(TABLES_DIR / "null_model_homophily.csv").set_index(
        "null_model"
    )
    null_modularity = pd.read_csv(TABLES_DIR / "null_model_modularity.csv").iloc[0]
    network_models = pd.read_csv(TABLES_DIR / "network_vs_random_models.csv")
    powerlaw = pd.read_csv(TABLES_DIR / "powerlaw_fit.csv").iloc[0]

    document = Document()
    _configure_styles(document)
    section = document.sections[0]
    _configure_section(section)
    _set_header_footer(section)

    # Page 1: integrated title, abstract and research questions.
    title = document.add_paragraph()
    _set_rtl(title, WD_ALIGN_PARAGRAPH.CENTER)
    title.paragraph_format.space_before = Pt(0)
    title.paragraph_format.space_after = Pt(2)
    run = title.add_run("תחום התמחות של מפתחי GitHub מתוך רשת מעקב הדדית")
    _set_run_font(run, size=18, bold=True, color=INK)
    subtitle = document.add_paragraph()
    _set_rtl(subtitle, WD_ALIGN_PARAGRAPH.CENTER)
    subtitle.paragraph_format.space_after = Pt(2)
    run = subtitle.add_run("פרויקט גמר — אלגוריתמים ברשתות")
    _set_run_font(run, size=10.5, bold=True, color=ACCENT)
    meta = document.add_paragraph()
    _set_rtl(meta, WD_ALIGN_PARAGRAPH.CENTER)
    meta.paragraph_format.space_after = Pt(5)
    run = meta.add_run(
        "נועה ירין לוי (208101196) · ג'ימי חלאוי (208301176) · "
        "אופיר אטיאס (211446935) | מרצה: ד\"ר אבנר פריאל"
    )
    _set_run_font(run, size=8.8, color=MUTED)

    _add_heading(document, "תקציר", 1)
    _add_paragraph(
        document,
        f"העבודה מנתחת רשת לא־מכוונת של {key['network_nodes']:,} מפתחי GitHub "
        f"ו־{key['network_edges']:,} קשרי מעקב הדדיים, ובוחנת ארבע שאלות: "
        "סיווג תחום מקצועי מתוך מבנה הרשת, ערך הרשת מול תוכן, שחזור קשתות "
        "שהוסתרו ומובהקות מבנית מול מודלי אפס. כל הניסויים חוזרים בחמש חלוקות "
        "קבועות; תוויות הבדיקה אינן משמשות בחישוב מאפיינים או בכיול. שיעור "
        f"הקשתות התוך־תחומיות הוא {_r(100 * key['homophily_rate'], 1)}%. "
        f"המודל החזק בסיווג הידני הגיע ל־AUC {_r(key['best_classification_roc_auc'])}; "
        f"DeepWalk ו־Node2Vec הגיעו ל־{_r(key['deepwalk_roc_auc'])} "
        f"ו־{_r(key['node2vec_roc_auc'])}. בשחזור קשתות, "
        f"{key['best_reconstruction_heuristic']} הוביל עם "
        f"AUC {_r(key['best_reconstruction_roc_auc'])}. "
        "התוצאות מתוארות כקשרים חיזויים בתנאי הניסוי ואינן מוכיחות סיבתיות."
    )
    _add_heading(document, "שאלות המחקר", 1)
    questions = [
        "RQ1: באיזו מידה שכנות, מרכזיות, קהילות וייצוגים נלמדים מאפשרים לסווג את תחום המפתח ללא תוויות test?",
        "RQ2: מהו הערך השולי של מבנה הרשת מול מאפייני תוכן, ומה מוסיף שילובם?",
        "RQ3: האם ניתן לשחזר קשתות מעקב שהוסתרו אקראית, והאם קשתות תוך־תחומיות קלות יותר לשחזור מקשתות בין־תחומיות?",
        "RQ4: האם ההומופיליה והמודולריות חריגות מול מודלי אפס, ואילו מאפיינים משחזרים מודלי ER, BA, Small-World ו־Configuration?",
    ]
    for question in questions:
        _add_paragraph(document, question, size=8.9, after=1.5)
    _add_heading(document, "הנתונים והיקף הניתוח", 1)
    _add_table(
        document,
        ["צמתים", "קשתות", "דרגה ממוצעת", "צפיפות", "רכיבי קשירות", "תוויות חסרות"],
        [[
            f"{int(basic['num_nodes']):,}",
            f"{int(basic['num_edges']):,}",
            _r(basic["average_degree"]),
            _r(basic["density"], 6),
            str(int(basic["num_connected_components"])),
            "0",
        ]],
        [1500, 1500, 1800, 1600, 1800, 2240],
        font_size=8.0,
    )

    # Page 2: methods and the exact ten logical classification models.
    _page_break(document)
    _add_heading(document, "1. מסגרת מתודולוגית ושיטות", 1)
    _add_paragraph(
        document,
        "בסיווג הצמתים מבנה הגרף גלוי לכל הצמתים (transductive), אך תוויות test "
        "מוסתרות. מאפייני שכנים וקהילה מחושבים מתוויות train בלבד. בכל seed "
        "מתבצעת חלוקה מרובדת 70/30; כיול סף משתמש ב־validation פנימי מתוך train.",
    )
    _add_heading(document, "ניתוח הרשת", 2)
    _add_paragraph(
        document,
        "נמדדו Degree Centrality, Clustering Coefficient, PageRank, Betweenness "
        "בקירוב, HITS Hub יחיד, Louvain ומודולריות. התפלגות הדרגות נבחנה בזהירות "
        "באמצעות התאמת Power Law והשוואה ללוג־נורמלית ולמעריכית. הרשת הושוותה "
        "ל־ER, BA, Watts-Strogatz ולגרף משמר־דרגות.",
    )
    _add_heading(document, "מודלי בסיס ופיצ'רים ידניים", 2)
    _add_table(
        document,
        ["מודל", "קלט", "מטרת ההשוואה"],
        [
            ["Majority Baseline", "שכיחות train", "קו בסיס ללא מבנה"],
            ["Neighbor Majority", "תוויות שכני train", "אות שכונתי מקומי"],
            ["LogReg Graph", "מרכזיות ומבנה", "פיצ'רים ידניים"],
            ["LogReg Graph + Community", "מבנה וקהילה", "ערך קהילה מעבר למבנה"],
        ],
        [3200, 3200, 4040],
        font_size=8.1,
    )
    _add_heading(document, "ייצוגים נלמדים ומודלי תוכן", 2)
    _add_table(
        document,
        ["מודל", "קלט", "תפקיד"],
        [
            ["LogReg Content", "מאפייני MUSAE", "נקודת ייחוס ללא רשת"],
            ["LogReg Hybrid", "רשת + תוכן", "מידע משלים"],
            ["LogReg DeepWalk", "הילוכים אחידים", "ייצוג מבני נלמד"],
            ["DeepWalk + ידני", "הטמעה + מבנה/קהילה", "שילוב ייצוגים"],
            ["LogReg Node2Vec", "הילוכים מוטים", "ייצוג מבני נלמד"],
            ["Node2Vec + ידני", "הטמעה + מבנה/קהילה", "שילוב ייצוגים"],
        ],
        [3200, 3200, 4040],
        font_size=7.9,
    )
    _add_heading(document, "שחזור קשתות ומודלי אפס", 2)
    _add_paragraph(
        document,
        "בשחזור קשתות מוגן עץ פורש וקשתות test נדגמות רק מן הקשתות האחרות; "
        "כך גרף האימון נשאר קשיר. non-edges מותאמים לפי Web-Web, ML-ML ו־Web-ML. "
        "נבדקים Common Neighbors, Jaccard, Adamic-Adar, Preferential Attachment, "
        "Same Community, DeepWalk ו־Node2Vec. ערכי p אמפיריים מחושבים עם "
        "(b+1)/(B+1), ולכן אינם אפס.",
    )

    # Page 3: structural analysis and communities.
    _page_break(document)
    _add_heading(document, "2. מבנה הרשת, מרכזיות וקהילות", 1)
    _add_paragraph(
        document,
        f"הרשת קשירה, צפופה מעט בלבד ({basic['density']:.6f}) ובעלת דרגת שיא "
        f"{int(pd.read_csv(TABLES_DIR / 'network_vs_random_models.csv').iloc[0]['max_degree']):,}. "
        "Degree ו־PageRank מדגישים מוקדי קשרים, בעוד Betweenness מדגישה תיווך "
        "בין אזורים; לכן טבלאות המובילים אינן זהות.",
    )
    top_rows = []
    for metric in ("Degree Centrality", "PageRank", "Betweenness Centrality"):
        row = centrality[centrality["metric"].str.startswith(metric)].sort_values("rank").iloc[0]
        top_rows.append(
            [
                metric.replace(" Centrality", ""),
                str(row["developer_name"]),
                str(int(row["node_id"])),
                f"{int(row['degree']):,}",
                f"{float(row['score']):.6f}",
            ]
        )
    _add_table(
        document,
        ["מדד", "מפתח מוביל", "Node ID", "דרגה", "ציון"],
        top_rows,
        [1800, 3000, 1500, 1300, 2840],
        font_size=8.0,
    )
    degree_pr = overlap[
        (
            overlap["metric_a"].str.startswith("Degree Centrality")
            & overlap["metric_b"].str.startswith("PageRank")
        )
        | (
            overlap["metric_b"].str.startswith("Degree Centrality")
            & overlap["metric_a"].str.startswith("PageRank")
        )
    ]["jaccard_overlap"].iloc[0]
    degree_bet = overlap[
        (
            overlap["metric_a"].str.startswith("Degree Centrality")
            & overlap["metric_b"].str.startswith("Betweenness")
        )
        | (
            overlap["metric_b"].str.startswith("Degree Centrality")
            & overlap["metric_a"].str.startswith("Betweenness")
        )
    ]["jaccard_overlap"].iloc[0]
    _add_paragraph(
        document,
        f"חפיפת Jaccard בין 20 המובילים ב־Degree וב־PageRank היא "
        f"{degree_pr:.3f}, לעומת {degree_bet:.3f} בין Degree וב־Betweenness. "
        "הפער ממחיש שמתווכים אינם בהכרח הצמתים בעלי מספר הקשרים הגדול ביותר.",
    )
    _add_picture(
        document,
        "community_size_and_composition.png",
        "איור 1. שתים־עשרה קהילות Louvain הגדולות: גודל והרכב תחומי.",
        width=6.15,
    )
    _add_paragraph(
        document,
        f"Louvain מצא {int(community_metrics['number_of_communities'])} קהילות "
        f"עם מודולריות {community_metrics['modularity']:.3f}. הקהילה הגדולה "
        f"מכילה {int(community_metrics['largest_community_size']):,} צמתים; "
        f"הטוהר הממוצע המשוקלל הוא {community_metrics['size_weighted_mean_purity']:.3f}. "
        "הרכבים שונים בין קהילות מצביעים על קשר בין מבנה לתחום, אך קהילות "
        "מעורבות מונעות פירוש דטרמיניסטי.",
    )

    # Page 4: classification.
    _page_break(document)
    _add_heading(document, "3. RQ1 — סיווג צמתים", 1)
    class_rows = []
    for _, row in classification.sort_values("roc_auc_mean").iterrows():
        class_rows.append(
            [
                str(row["model"]).replace("Logistic Regression - ", "LogReg "),
                _r(row["macro_f1_mean"]),
                _r(row["macro_f1_std"]),
                _r(row["roc_auc_mean"]),
                _r(row["roc_auc_std"]),
            ]
        )
    _add_table(
        document,
        ["מודל", "Macro F1", "סטיית תקן", "ROC-AUC", "סטיית תקן"],
        class_rows,
        [4100, 1600, 1500, 1600, 1640],
        font_size=8.0,
    )
    _add_picture(
        document,
        "classification_comparison.png",
        "איור 2. ביצועי סיווג הצמתים על פני חמש חלוקות; פסי השגיאה הם סטיית תקן.",
        width=6.25,
    )
    default_threshold = thresholds.loc["default_threshold_0.5"]
    tuned_threshold = thresholds.loc["tuned_threshold_validation_f1"]
    _add_paragraph(
        document,
        f"הוספת קהילה העלתה את ROC-AUC ל־"
        f"{key['best_classification_roc_auc']:.3f}. סף שנבחר על validation בלבד "
        f"היה {tuned_threshold['selected_threshold_mean']:.3f} בממוצע; הוא שינה "
        f"את Precision מ־{default_threshold['precision_mean']:.3f} ל־"
        f"{tuned_threshold['precision_mean']:.3f} ואת Recall מ־"
        f"{default_threshold['recall_mean']:.3f} ל־{tuned_threshold['recall_mean']:.3f}. "
        "השינוי הוא trade-off תפעולי ואינו שיפור מובטח בכל מדד.",
    )
    _add_paragraph(
        document,
        "מסקנת RQ1: מבנה הרשת מספק אות חיזויי מעל קווי הבסיס; קהילה מוסיפה "
        "מידע מעבר לשכנות ולמרכזיות. המסקנה מוגבלת לפרוטוקול transductive ולתווית "
        "הבינארית שנבדקה.",
        bold_lead="מסקנת RQ1:",
    )

    # Page 5: ablation and embeddings.
    _page_break(document)
    _add_heading(document, "4. RQ2 — רשת, תוכן וייצוגים נלמדים", 1)
    comparison_rows = []
    for _, row in ablation.iterrows():
        comparison_rows.append(
            [
                str(row["model"]).replace("Logistic Regression - ", "LogReg "),
                _r(row["macro_f1_mean"]),
                _r(row["roc_auc_mean"]),
                _r(row["roc_auc_std"]),
            ]
        )
    for _, row in embeddings.iterrows():
        comparison_rows.append(
            [
                str(row["model"]).replace("Logistic Regression - ", "LogReg "),
                _r(row["macro_f1_mean"]),
                _r(row["roc_auc_mean"]),
                _r(row["roc_auc_std"]),
            ]
        )
    _add_table(
        document,
        ["ייצוג/מקור מידע", "Macro F1", "ROC-AUC", "סטיית תקן AUC"],
        comparison_rows,
        [5600, 1600, 1600, 1640],
        font_size=7.8,
    )
    _add_picture(
        document,
        "representation_comparison.png",
        "איור 3. השוואה הוגנת בין פיצ'רים ידניים, תוכן, DeepWalk, Node2Vec והיברידי.",
        width=6.3,
    )
    hybrid = ablation[ablation["model"].str.contains("Hybrid", regex=False)].iloc[0]
    content = ablation[ablation["model"].str.contains("Content Features Only", regex=False)].iloc[0]
    network_only = ablation[ablation["model"].str.contains("Network Only", regex=False)].iloc[0]
    _add_paragraph(
        document,
        f"תוכן לבדו הגיע ל־AUC {content['roc_auc_mean']:.3f}, מול "
        f"{network_only['roc_auc_mean']:.3f} בפיצ'רים ידניים של הרשת ו־"
        f"{hybrid['roc_auc_mean']:.3f} בשילוב. DeepWalk לבדו הגיע ל־"
        f"{key['deepwalk_roc_auc']:.3f} ו־Node2Vec ל־{key['node2vec_roc_auc']:.3f}. "
        "הדמיון בביצועים מצביע על חפיפה חיזויית בתנאי הניסוי; הוא אינו מוכיח "
        "שקילות בין מקורות המידע.",
    )
    _add_paragraph(
        document,
        "מסקנת RQ2: פיצ'רי תוכן חזקים מן הייצוג הידני של הרשת, אך השילוב משפר "
        "את התוצאה. הטמעות הילוכים מרכזות מידע מבני שימושי ומצמצמות חלק מן הפער.",
        bold_lead="מסקנת RQ2:",
    )

    # Page 6: edge reconstruction.
    _page_break(document)
    _add_heading(document, "5. RQ3 — שחזור קשתות קשיר", 1)
    _add_paragraph(
        document,
        "בכל seed מוגנות כל קשתות עץ פורש, ולכן הסרת 10% מהקשתות שאינן בעץ "
        "אינה מפרקת את גרף האימון. בכל חמש החלוקות התקבל רכיב קשיר אחד וללא "
        "צמתים מבודדים. שמירה זו נדרשת במיוחד למסלולים, Louvain והטמעות המבוססות "
        "הילוכים. לכל positive הותאם non-edge מאותו סוג קצוות.",
    )
    reconstruction_rows = []
    for _, row in reconstruction.sort_values("roc_auc_mean", ascending=False).iterrows():
        reconstruction_rows.append(
            [
                str(row["heuristic"]),
                _r(row["roc_auc_mean"]),
                _r(row["roc_auc_std"]),
                _r(row["average_precision_mean"]),
                _r(row["average_precision_std"]),
            ]
        )
    _add_table(
        document,
        ["שיטה", "ROC-AUC", "סטיית תקן", "Average Precision", "סטיית תקן"],
        reconstruction_rows,
        [3700, 1500, 1400, 2100, 1740],
        font_size=7.7,
    )
    _add_picture(
        document,
        "edge_reconstruction_same_vs_cross.png",
        "איור 4. השוואה ישירה same-field מול cross-field תחת דגימה מאוזנת.",
        width=6.35,
    )
    best_group = reconstruction_groups[
        reconstruction_groups["heuristic"] == key["best_reconstruction_heuristic"]
    ].set_index("edge_group")
    same_auc = best_group.loc["same-field", "roc_auc_mean"]
    cross_auc = best_group.loc["cross-field", "roc_auc_mean"]
    same_ap = best_group.loc["same-field", "average_precision_mean"]
    cross_ap = best_group.loc["cross-field", "average_precision_mean"]
    direction = "גבוה" if same_auc > cross_auc else "נמוך"
    _add_paragraph(
        document,
        f"{key['best_reconstruction_heuristic']} הוביל עם AUC "
        f"{key['best_reconstruction_roc_auc']:.3f}. באותה שיטה, same-field קיבל "
        f"AUC {same_auc:.3f} ו־AP {same_ap:.3f}, לעומת AUC {cross_auc:.3f} "
        f"ו־AP {cross_ap:.3f} ב־cross-field. לכן השחזור התוך־תחומי {direction} "
        "יותר בפרוטוקול זה. התוצאה מתארת יכולת שחזור ואינה מוכיחה מנגנון סיבתי.",
    )

    # Page 7: null models and network-model comparison.
    _page_break(document)
    _add_heading(document, "6. RQ4 — מודלי אפס ומודלי רשת", 1)
    _add_table(
        document,
        ["מודל אפס", "נצפה", "ממוצע אפס", "סטיית תקן", "p אמפירי"],
        [
            [
                "ערבוב תוויות",
                _r(null_homophily.loc["label_shuffle", "observed"]),
                _r(null_homophily.loc["label_shuffle", "null_mean"]),
                _r(null_homophily.loc["label_shuffle", "null_std"]),
                _r(null_homophily.loc["label_shuffle", "empirical_p"], 6),
            ],
            [
                "גרף משמר־דרגות",
                _r(null_homophily.loc["degree_preserving_rewire", "observed"]),
                _r(null_homophily.loc["degree_preserving_rewire", "null_mean"]),
                _r(null_homophily.loc["degree_preserving_rewire", "null_std"]),
                _r(null_homophily.loc["degree_preserving_rewire", "empirical_p"], 6),
            ],
        ],
        [2800, 1600, 2000, 1800, 2240],
        font_size=8.0,
    )
    model_rows = []
    for _, row in network_models.iterrows():
        model_rows.append(
            [
                str(row["network"]),
                f"{int(row['edges']):,}",
                _r(row["avg_clustering"], 4),
                _r(row["avg_shortest_path_sampled"]),
                str(int(row["max_degree"])),
            ]
        )
    _add_table(
        document,
        ["רשת", "קשתות", "Clustering", "מרחק ממוצע", "דרגת שיא"],
        model_rows,
        [4000, 1800, 1600, 1800, 1240],
        font_size=7.5,
    )
    _add_picture(
        document,
        "homophily_null_distribution.png",
        "איור 5. הומופיליה נצפית מול ערבוב תוויות וגרפים משמרי־דרגות.",
        width=5.9,
    )
    _add_paragraph(
        document,
        f"המודולריות הנצפית היא {null_modularity['observed_modularity']:.3f}, "
        f"מול {null_modularity['rewired_null_mean']:.3f} בממוצע בגרפים "
        f"משמרי־דרגות (p={null_modularity['empirical_p']:.6f}). Power Law נותן "
        f"α={powerlaw['alpha']:.2f}; ההשוואה ללוג־נורמלית אינה מספקת בסיס "
        "להכרזה גורפת על scale-free, ולכן המסקנה הזהירה היא זנב כבד. ER אינו "
        "משחזר התקבצות או דרגת שיא; BA משחזר חלק מן הזנב; Small-World משחזר "
        "התקבצות ומרחקים קצרים; הגרף משמר־הדרגות מבודד את תרומת רצף הדרגות.",
    )

    # Page 8: synthesis, limitations, conclusions and compact references.
    _page_break(document)
    _add_heading(document, "7. דיון מסכם ומגבלות", 1)
    _add_paragraph(
        document,
        "התמונה המצטברת עקבית בין ארבע השאלות: יש קשר תיאורי חזק בין מבנה "
        "הרשת לתחום, קהילות והטמעות מוסיפות כוח חיזויי, ותוכן מספק אות חזק "
        "נוסף. מודלי האפס מראים שרצף הדרגות מסביר חלק מן ההומופיליה אך לא את "
        "כולה. בשחזור הקשתות, הצלחת Preferential Attachment עקבית עם חשיבות "
        "הדרגות, אך אינה מוכיחה שהעדפת צמתים בעלי דרגה גבוהה היא מנגנון יצירת הקשר.",
    )
    _add_heading(document, "מגבלות", 2)
    limitations = [
        "הנתונים הם תמונת מצב מיוני 2019 ואינם מייצגים בהכרח את GitHub כיום.",
        "תווית היעד בינארית ונגזרה ממידע פרופיל; ייתכן רעש וחסר בתחום המקצועי.",
        "הסיווג transductive, ולכן אינו מדמה צומת חדש ללא קשרים ידועים.",
        "Betweenness מחושב בקירוב, והטמעות נבדקו בתקציב פרמטרים קבוע ללא חיפוש רחב.",
        "שחזור קשתות שהוסתרו אינו תחזית זמן ואינו מזהה סיבתיות או כיווניות.",
        "מודלי האפס והמודלים הגנרטיביים מבודדים מאפיינים מסוימים ואינם תיאור מלא של תהליך הרשת.",
    ]
    for item in limitations:
        _add_bullet(document, item)
    _add_heading(document, "מסקנות", 2)
    _add_paragraph(
        document,
        "RQ1 נענה בחיוב מוגבל: מבנה הרשת מאפשר סיווג מעל קווי הבסיס. RQ2 מראה "
        "חפיפה חלקית ומידע משלים בין רשת לתוכן. RQ3 מראה ששחזור קשתות אפשרי "
        "תוך שמירת קשירות, והפער same/cross תלוי בשיטה ובפרוטוקול. RQ4 מראה "
        "שההומופיליה והמודולריות חריגות ביחס למודלי האפס שנבדקו. הממצאים "
        "תומכים בערך האנליטי של רשתות מקצועיות, תוך שמירה על ניסוח תיאורי וזהיר.",
    )
    _add_heading(document, "מקורות", 2)
    references = [
        "Rozemberczki, Allen & Sarkar (2019), MUSAE.",
        "Blondel et al. (2008), Fast unfolding of communities.",
        "Perozzi, Al-Rfou & Skiena (2014), DeepWalk.",
        "Grover & Leskovec (2016), node2vec.",
        "Liben-Nowell & Kleinberg (2003), Link Prediction.",
        "Clauset, Shalizi & Newman (2009), Power-law distributions.",
        "Page et al. (1999), PageRank.",
        "מצגות ותרגולי הקורס: מרכזיות, קהילות, מודלי רשת, שחזור קשתות והטמעות.",
    ]
    for index, reference in enumerate(references, start=1):
        _add_paragraph(document, f"{index}. {reference}", size=7.8, after=0.6)

    # Ensure all sections (including any Word-created continuation sections)
    # use the same compact page geometry and furniture.
    for section in document.sections:
        _configure_section(section)
        _set_header_footer(section)

    document.core_properties.title = (
        "תחום התמחות של מפתחי GitHub מתוך רשת מעקב הדדית"
    )
    document.core_properties.subject = "Final project - network algorithms"
    document.core_properties.author = (
        "Noa Yarin Levy; Jimi Halawi; Ofir Atias"
    )
    document.save(REPORT_PATH)
    print(f"Built report source: {REPORT_PATH}")




**פלט ומשמעות.** בלוק ``main`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report.py, שורות 842-843: `__main__ guard`

**מה ומדוע.** זהו שער ההרצה המקורי של קובץ CLI. במחברת הוא נשמר כלשונו, אך אינו מופעל כי `__name__` הוגדר כשם מחברת; ההרצה נעשית בתא מפורש ומתועד.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** אין פלט, משום ששער ה-CLI מושבת בכוונה בסביבת המחברת.

In [60]:
if __name__ == "__main__":
    main()


**פלט ומשמעות.** בלוק ``__main__ guard`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

## קוד המקור: `src/build_report_pdf.py`

רינדור PDF שחזורי ישירות מהטבלאות והתרשימים, ללא תלות ב-Office.

לפני טעינת הקטעים נגדיר עבורם `__file__` וירטואלי התואם למיקומם המקורי. הדבר משמר את לוגיקת חישוב `PROJECT_DIR` בלי לשנות שורה בקוד המקור. SHA-256 של המקור: `5496f202db2263fedd4c2132b397437d8bab30dcbdd1320229e46b270a8509a8`.

התא הבא מכין את הקשר הנתיבים עבור קובץ המקור. הוא אינו משנה את הקובץ; הפלט הצפוי הוא נתיב המקור שממנו הועתק הקוד.

In [61]:
__file__ = str(ROOT / 'src/build_report_pdf.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\build_report_pdf.py


**פלט ומשמעות.** הנתיב המודפס מאפשר לבקר שהקבועים בקוד יחושבו ביחס לריפוזיטורי הנוכחי בלבד.

### build_report_pdf.py, שורות 1-45: ייבואים, קבועים והגדרות מודול

**מה ומדוע.** הבלוק מרכז את התלויות, הקבועים והנחות ברירת המחדל של המודול; בלעדיו הפונקציות הבאות אינן עצמאיות.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [62]:
# -*- coding: utf-8 -*-
"""Build the final eight-page PDF directly from the fresh result tables.

The editable Word source is produced by ``build_report.py``.  This renderer
uses the same tables and narrative, but avoids depending on an installed
office-suite PDF driver so that the final artifact is reproducible on Windows,
Linux and macOS.
"""

from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from bidi.algorithm import get_display
from PIL import Image
from reportlab.lib.colors import HexColor, white
from reportlab.lib.pagesizes import LETTER
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen.canvas import Canvas

SRC_DIR = Path(__file__).resolve().parent
PROJECT_DIR = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))

from final_validation import collect_key_results


REPORT_PATH = PROJECT_DIR / "report" / "final_report.pdf"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"

PAGE_W, PAGE_H = LETTER
MARGIN = 38
CONTENT_W = PAGE_W - 2 * MARGIN
INK = HexColor("#172B4D")
ACCENT = HexColor("#315C8D")
ORANGE = HexColor("#D57837")
MUTED = HexColor("#5F6B7A")
LIGHT = HexColor("#E8EEF5")
PALE = HexColor("#F4F6F9")




**פלט ומשמעות.** בלוק `ייבואים, קבועים והגדרות מודול` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 46-71: `_font_paths`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [63]:
def _font_paths() -> tuple[Path, Path]:
    candidates = [
        (
            Path("C:/Windows/Fonts/arial.ttf"),
            Path("C:/Windows/Fonts/arialbd.ttf"),
        ),
        (
            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"),
        ),
        (
            Path("/Library/Fonts/Arial.ttf"),
            Path("/Library/Fonts/Arial Bold.ttf"),
        ),
    ]
    for regular, bold in candidates:
        if regular.is_file() and bold.is_file():
            return regular, bold
    raise FileNotFoundError("A Hebrew-capable Arial or DejaVu font is required.")


REGULAR_FONT, BOLD_FONT = _font_paths()
pdfmetrics.registerFont(TTFont("Report", str(REGULAR_FONT)))
pdfmetrics.registerFont(TTFont("Report-Bold", str(BOLD_FONT)))




**פלט ומשמעות.** בלוק ``_font_paths`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 72-75: `_visual`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [64]:
def _visual(text: object) -> str:
    return get_display(str(text), base_dir="R")




**פלט ומשמעות.** בלוק ``_visual`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 76-92: `_wrap`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [65]:
def _wrap(text: str, width: float, font: str, size: float) -> list[str]:
    words = str(text).split()
    if not words:
        return [""]
    lines: list[str] = []
    current = words[0]
    for word in words[1:]:
        candidate = f"{current} {word}"
        if pdfmetrics.stringWidth(_visual(candidate), font, size) <= width:
            current = candidate
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines




**פלט ומשמעות.** בלוק ``_wrap`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 93-104: `_shorten`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [66]:
def _shorten(text: object, width: float, font: str, size: float) -> str:
    logical = str(text)
    if pdfmetrics.stringWidth(_visual(logical), font, size) <= width:
        return logical
    suffix = "…"
    while logical and pdfmetrics.stringWidth(
        _visual(logical + suffix), font, size
    ) > width:
        logical = logical[:-1]
    return logical + suffix




**פלט ומשמעות.** בלוק ``_shorten`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 105-251: `ReportCanvas`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [67]:
class ReportCanvas:
    def __init__(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        self.canvas = Canvas(str(path), pagesize=LETTER)
        self.canvas.setTitle(
            "תחום ההתמחות של מפתחי GitHub מתוך רשת מעקב הדדית"
        )
        self.canvas.setAuthor("Noa Yarin Levy; Jimi Halawi; Ofir Atias")
        self.y = PAGE_H - 58
        self.page_number = 0

    def begin_page(self, section: str) -> None:
        self.page_number += 1
        self.y = PAGE_H - 55
        c = self.canvas
        c.setStrokeColor(LIGHT)
        c.setLineWidth(0.8)
        c.line(MARGIN, PAGE_H - 40, PAGE_W - MARGIN, PAGE_H - 40)
        c.setFillColor(MUTED)
        c.setFont("Report", 7.4)
        c.drawRightString(
            PAGE_W - MARGIN,
            PAGE_H - 32,
            _visual("תחום התמחות ברשת GitHub | פרויקט גמר"),
        )
        c.setFillColor(ACCENT)
        c.setFont("Report-Bold", 7.4)
        c.drawString(MARGIN, PAGE_H - 32, _visual(section))

    def finish_page(self) -> None:
        c = self.canvas
        c.setStrokeColor(LIGHT)
        c.line(MARGIN, 30, PAGE_W - MARGIN, 30)
        c.setFillColor(MUTED)
        c.setFont("Report", 7.5)
        c.drawCentredString(
            PAGE_W / 2,
            18,
            _visual(f"עמוד {self.page_number} מתוך 8"),
        )
        c.showPage()

    def heading(self, text: str, *, size: float = 14, gap: float = 5) -> None:
        self.canvas.setFillColor(ACCENT)
        self.canvas.setFont("Report-Bold", size)
        self.canvas.drawRightString(PAGE_W - MARGIN, self.y, _visual(text))
        self.y -= size + gap

    def paragraph(
        self,
        text: str,
        *,
        size: float = 9.0,
        leading: float = 11.5,
        gap: float = 5,
        bold: bool = False,
        color=INK,
        indent: float = 0,
    ) -> None:
        font = "Report-Bold" if bold else "Report"
        self.canvas.setFillColor(color)
        self.canvas.setFont(font, size)
        for line in _wrap(text, CONTENT_W - indent, font, size):
            self.canvas.drawRightString(
                PAGE_W - MARGIN - indent, self.y, _visual(line)
            )
            self.y -= leading
        self.y -= gap

    def bullet(self, text: str) -> None:
        self.paragraph(f"• {text}", size=8.4, leading=10.2, gap=1.5, indent=8)

    def table(
        self,
        headers: list[str],
        rows: list[list[object]],
        widths: list[float],
        *,
        font_size: float = 7.6,
        row_height: float = 21,
        gap: float = 6,
    ) -> None:
        if abs(sum(widths) - CONTENT_W) > 0.1:
            raise ValueError("Table widths must sum to the content width.")
        c = self.canvas
        x0 = MARGIN
        y_top = self.y
        all_rows = [headers, *rows]
        for row_index, values in enumerate(all_rows):
            y = y_top - row_height * (row_index + 1)
            c.setFillColor(LIGHT if row_index == 0 else (PALE if row_index % 2 == 0 else white))
            c.rect(x0, y, CONTENT_W, row_height, fill=1, stroke=0)
            x = x0
            for width, value in zip(widths, values):
                c.setStrokeColor(white)
                c.rect(x, y, width, row_height, fill=0, stroke=1)
                font = "Report-Bold" if row_index == 0 else "Report"
                c.setFont(font, font_size)
                c.setFillColor(INK)
                shown = _shorten(value, width - 8, font, font_size)
                c.drawRightString(
                    x + width - 4,
                    y + (row_height - font_size) / 2 + 1.5,
                    _visual(shown),
                )
                x += width
        self.y = y_top - row_height * len(all_rows) - gap

    def figure(
        self,
        filename: str,
        caption: str,
        *,
        box_height: float,
        gap: float = 6,
    ) -> None:
        path = FIGURES_DIR / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        with Image.open(path) as image:
            image_w, image_h = image.size
        scale = min(CONTENT_W / image_w, box_height / image_h)
        draw_w = image_w * scale
        draw_h = image_h * scale
        x = MARGIN + (CONTENT_W - draw_w) / 2
        y = self.y - draw_h
        self.canvas.drawImage(
            str(path),
            x,
            y,
            width=draw_w,
            height=draw_h,
            preserveAspectRatio=True,
            mask="auto",
        )
        self.y = y - 10
        self.canvas.setFillColor(MUTED)
        self.canvas.setFont("Report", 7.2)
        self.canvas.drawCentredString(PAGE_W / 2, self.y, _visual(caption))
        self.y -= 8 + gap

    def save(self) -> None:
        if self.page_number != 8:
            raise AssertionError(f"Expected eight pages, got {self.page_number}.")
        self.canvas.save()




**פלט ומשמעות.** בלוק ``ReportCanvas`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 252-255: `_r`

**מה ומדוע.** הבלוק מגדיר יחידת חישוב חוזרת בצנרת ומפריד בין לוגיקה לבין תזמור הניסוי.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [68]:
def _r(value: float, digits: int = 3) -> str:
    return f"{float(value):.{digits}f}"




**פלט ומשמעות.** בלוק ``_r`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 256-729: `main`

**מה ומדוע.** פונקציית התזמור של הקובץ. היא מוגדרת כאן אך אינה מופעלת אוטומטית; תא הרצה מפורש בהמשך מפעיל אותה ושומר פלטים.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** הגדרה בזיכרון ללא פלט חזותי; התוצאה תופיע כאשר הפונקציה תיקרא בתא הניסוי.

In [69]:
def main() -> None:
    key = collect_key_results(PROJECT_DIR)
    basic = pd.read_csv(TABLES_DIR / "network_basic_stats.csv").iloc[0]
    centrality = pd.read_csv(TABLES_DIR / "centrality_top_developers.csv")
    overlap = pd.read_csv(TABLES_DIR / "centrality_top_overlap.csv")
    community = pd.read_csv(TABLES_DIR / "community_metrics.csv").iloc[0]
    classification = pd.read_csv(TABLES_DIR / "classification_summary.csv")
    thresholds = pd.read_csv(TABLES_DIR / "threshold_tuning_summary.csv").set_index(
        "evaluation_type"
    )
    ablation = pd.read_csv(TABLES_DIR / "ablation_model_results_summary.csv")
    embeddings = pd.read_csv(TABLES_DIR / "embedding_classification_summary.csv")
    reconstruction = pd.read_csv(TABLES_DIR / "edge_reconstruction_summary.csv")
    recon_groups = pd.read_csv(
        TABLES_DIR / "edge_reconstruction_same_vs_cross_summary.csv"
    )
    null_h = pd.read_csv(TABLES_DIR / "null_model_homophily.csv").set_index(
        "null_model"
    )
    null_m = pd.read_csv(TABLES_DIR / "null_model_modularity.csv").iloc[0]
    networks = pd.read_csv(TABLES_DIR / "network_vs_random_models.csv")
    powerlaw = pd.read_csv(TABLES_DIR / "powerlaw_fit.csv").iloc[0]

    report = ReportCanvas(REPORT_PATH)

    # Page 1 — title, abstract, research questions and data.
    report.begin_page("תקציר ושאלות מחקר")
    report.canvas.setFillColor(INK)
    report.canvas.setFont("Report-Bold", 18)
    report.canvas.drawCentredString(
        PAGE_W / 2,
        report.y,
        _visual("תחום ההתמחות של מפתחי GitHub מתוך רשת מעקב הדדית"),
    )
    report.y -= 23
    report.canvas.setFillColor(ACCENT)
    report.canvas.setFont("Report-Bold", 10)
    report.canvas.drawCentredString(
        PAGE_W / 2, report.y, _visual("פרויקט גמר — אלגוריתמים ברשתות")
    )
    report.y -= 16
    report.canvas.setFillColor(MUTED)
    report.canvas.setFont("Report", 8.2)
    report.canvas.drawCentredString(
        PAGE_W / 2,
        report.y,
        _visual(
            "נועה ירין לוי (208101196) · ג'ימי חלאוי (208301176) · אופיר אטיאס (211446935)"
        ),
    )
    report.y -= 12
    report.canvas.drawCentredString(
        PAGE_W / 2, report.y, _visual('מרצה: ד"ר אבנר פריאל')
    )
    report.y -= 20
    report.heading("תקציר", size=13)
    report.paragraph(
        f"העבודה מנתחת רשת לא־מכוונת של {key['network_nodes']:,} מפתחי GitHub "
        f"ו־{key['network_edges']:,} קשרי מעקב הדדיים. נבדקו סיווג תחום מקצועי "
        "ממבנה הרשת, הערך המשלים של תוכן, שחזור מדויק של קשתות שהוסתרו ומובהקות "
        "מבנית מול מודלי אפס. כל ניסויי החיזוי חזרו בחמש חלוקות קבועות, ללא שימוש "
        "בתוויות הבדיקה לבניית מאפיינים או לכיול. "
        f"שיעור הקשתות התוך־תחומיות הוא {100 * key['homophily_rate']:.1f}%; "
        f"המסווג הידני הטוב הגיע ל־ROC-AUC {key['best_classification_roc_auc']:.3f}; "
        f"DeepWalk ו־Node2Vec הגיעו ל־{key['deepwalk_roc_auc']:.3f} "
        f"ו־{key['node2vec_roc_auc']:.3f}. בשחזור קשתות הובילה "
        f"{key['best_reconstruction_heuristic']} עם ROC-AUC "
        f"{key['best_reconstruction_roc_auc']:.3f}. אלו קשרים חיזויים בתנאי "
        "הניסוי ואינם ראיה לסיבתיות."
    )
    report.heading("שאלות המחקר", size=12)
    report.paragraph(
        "RQ1: באיזו מידה שכנות, מרכזיות, קהילות וייצוגי הליכות מאפשרים לסווג את "
        "תחום המפתח כאשר תוויות הבדיקה מוסתרות?",
        size=8.6,
        leading=10.6,
        gap=2,
    )
    report.paragraph(
        "RQ2: מהו הערך השולי של מבנה הרשת מול מאפייני תוכן, ומה מוסיף השילוב?",
        size=8.6,
        leading=10.6,
        gap=2,
    )
    report.paragraph(
        "RQ3: האם ניתן לשחזר קשתות שהוסתרו אקראית תוך שמירת קישוריות, והאם "
        "same-field קל יותר לשחזור מ־cross-field?",
        size=8.6,
        leading=10.6,
        gap=2,
    )
    report.paragraph(
        "RQ4: האם הומופיליה ומודולריות חריגות מול מודלי אפס, ואילו מאפיינים "
        "משחזרים ER, BA, Small-World ורשת משמרת־דרגה?",
        size=8.6,
        leading=10.6,
        gap=5,
    )
    report.table(
        ["צמתים", "קשתות", "דרגה ממוצעת", "צפיפות", "רכיבי קישוריות"],
        [[
            f"{int(basic['num_nodes']):,}",
            f"{int(basic['num_edges']):,}",
            _r(basic["average_degree"]),
            _r(basic["density"], 6),
            str(int(basic["num_connected_components"])),
        ]],
        [92, 100, 115, 110, CONTENT_W - 417],
        font_size=7.7,
    )
    report.finish_page()

    # Page 2 — methodology and exact logical model inventory.
    report.begin_page("1 · מתודולוגיה ושיטות")
    report.heading("מסגרת מתודולוגית ושיטות")
    report.paragraph(
        "בסיווג הצמתים מבנה הגרף גלוי (transductive), אך תוויות test מוסתרות. "
        "מאפייני שכנים וקהילה מחושבים מתוויות train בלבד. בכל seed בוצעה חלוקה "
        "מרובדת 70/30, וסף ההחלטה כויל רק על validation פנימי מתוך train."
    )
    report.paragraph(
        "ניתוח הרשת כולל Degree, Clustering, PageRank, Betweenness בקירוב, HITS "
        "Hub יחיד, Louvain ומודולריות. התפלגות הדרגות נבחנת בזהירות מול חלופות. "
        "לשחזור קשתות מוגן עץ פורש; non-edges מותאמים לפי Web-Web, ML-ML ו־Web-ML."
    )
    model_rows = [
        ["בסיס", "Majority", "שכיחות train"],
        ["בסיס", "Neighbor Majority", "תוויות שכני train"],
        ["ידני", "LogReg Graph", "מרכזיות ומבנה"],
        ["ידני", "LogReg Graph + Community", "מבנה וקהילה"],
        ["תוכן", "LogReg Content", "מאפייני MUSAE"],
        ["היברידי", "LogReg Hybrid", "רשת ותוכן"],
        ["הטמעה", "LogReg DeepWalk", "הליכות אחידות"],
        ["הטמעה", "DeepWalk + ידני", "הטמעה ומבנה"],
        ["הטמעה", "LogReg Node2Vec", "הליכות מוטות"],
        ["הטמעה", "Node2Vec + ידני", "הטמעה ומבנה"],
    ]
    report.table(
        ["משפחה", "מודל", "קלט מרכזי"],
        model_rows,
        [90, 225, CONTENT_W - 315],
        font_size=7.5,
        row_height=19,
    )
    report.heading("שחזור קשתות ומודלי אפס", size=11)
    report.paragraph(
        "השיטות הן Common Neighbors, Jaccard, Adamic-Adar, Preferential "
        "Attachment, Same Community, DeepWalk ו־Node2Vec. ההטמעות והקהילות "
        "נלמדות מחדש על גרף האימון בלבד. ערכי p אמפיריים מחושבים כ־(b+1)/(B+1), "
        "ולכן אינם אפס. ההשוואות כוללות 1,000 ערבובי תוויות, 20 חיווטים "
        "משמרי־דרגה ומודלי ER, BA ו־Watts-Strogatz."
    )
    report.finish_page()

    # Page 3 — structure and communities.
    report.begin_page("2 · מבנה, מרכזיות וקהילות")
    report.heading("מבנה הרשת, מרכזיות וקהילות")
    report.paragraph(
        f"הרשת קשירה וצפופה מעט בלבד ({basic['density']:.6f}). מדדי Degree "
        "ו־PageRank מדגישים מוקדי קשרים, בעוד Betweenness מדגיש תיווך בין אזורים; "
        "לכן רשימות המובילים אינן זהות."
    )
    top_rows: list[list[object]] = []
    for metric in ("Degree Centrality", "PageRank", "Betweenness Centrality"):
        row = (
            centrality[centrality["metric"].str.startswith(metric)]
            .sort_values("rank")
            .iloc[0]
        )
        top_rows.append(
            [
                metric.replace(" Centrality", ""),
                row["developer_name"],
                int(row["node_id"]),
                int(row["degree"]),
                f"{float(row['score']):.6f}",
            ]
        )
    report.table(
        ["מדד", "מפתח מוביל", "Node ID", "דרגה", "ציון"],
        top_rows,
        [100, 190, 76, 60, CONTENT_W - 426],
        font_size=7.2,
        row_height=20,
    )
    degree_pr = overlap[
        overlap.apply(
            lambda row: {
                row["metric_a"].replace(" Centrality", ""),
                row["metric_b"].replace(" Centrality", ""),
            }
            == {"Degree", "PageRank"},
            axis=1,
        )
    ]["jaccard_overlap"].iloc[0]
    degree_bet = overlap[
        overlap.apply(
            lambda row: "Degree" in row["metric_a"]
            + row["metric_b"]
            and "Betweenness" in row["metric_a"] + row["metric_b"],
            axis=1,
        )
    ]["jaccard_overlap"].iloc[0]
    report.paragraph(
        f"חפיפת Jaccard בין 20 המובילים ב־Degree וב־PageRank היא {degree_pr:.3f}, "
        f"לעומת {degree_bet:.3f} בין Degree ל־Betweenness. הפער ממחיש שמתווכים "
        "אינם בהכרח בעלי מספר הקשרים הגדול ביותר.",
        size=8.5,
    )
    report.figure(
        "community_size_and_composition.png",
        "איור 1. קהילות Louvain הגדולות: גודל והרכב תחומי.",
        box_height=260,
    )
    report.paragraph(
        f"Louvain מצא {int(community['number_of_communities'])} קהילות עם "
        f"מודולריות {community['modularity']:.3f}; הקהילה הגדולה מכילה "
        f"{int(community['largest_community_size']):,} צמתים והטוהר המשוקלל "
        f"הוא {community['size_weighted_mean_purity']:.3f}. קהילות מעורבות "
        "מונעות פירוש דטרמיניסטי של השתייכות קהילתית.",
        size=8.5,
    )
    report.finish_page()

    # Page 4 — node classification.
    report.begin_page("3 · RQ1 — סיווג צמתים")
    report.heading("RQ1 — סיווג צמתים")
    class_rows = [
        [
            str(row["model"]).replace("Logistic Regression - ", "LogReg "),
            _r(row["macro_f1_mean"]),
            _r(row["macro_f1_std"]),
            _r(row["roc_auc_mean"]),
            _r(row["roc_auc_std"]),
        ]
        for _, row in classification.sort_values("roc_auc_mean").iterrows()
    ]
    report.table(
        ["מודל", "Macro F1", "SD", "ROC-AUC", "SD"],
        class_rows,
        [230, 82, 58, 82, CONTENT_W - 452],
        font_size=7.4,
        row_height=20,
    )
    report.figure(
        "classification_comparison.png",
        "איור 2. סיווג צמתים בחמש חלוקות; פסי השגיאה הם סטיית תקן.",
        box_height=315,
    )
    default = thresholds.loc["default_threshold_0.5"]
    tuned = thresholds.loc["tuned_threshold_validation_f1"]
    report.paragraph(
        f"הוספת קהילה העלתה את ROC-AUC ל־{key['best_classification_roc_auc']:.3f}. "
        f"הסף שנבחר על validation בלבד היה {tuned['selected_threshold_mean']:.3f} "
        f"בממוצע. Precision השתנה מ־{default['precision_mean']:.3f} ל־"
        f"{tuned['precision_mean']:.3f}, ו־Recall מ־{default['recall_mean']:.3f} "
        f"ל־{tuned['recall_mean']:.3f}; זהו trade-off תפעולי ולא שיפור בכל מדד."
    )
    report.paragraph(
        "מסקנת RQ1: מבנה הרשת מספק אות חיזויי מעל קווי הבסיס, וקהילה מוסיפה "
        "מידע מעבר לשכנות ולמרכזיות. המסקנה מוגבלת לפרוטוקול transductive "
        "ולתווית הבינארית שנבדקה.",
        bold=True,
        size=8.6,
    )
    report.finish_page()

    # Page 5 — content ablation and learned representations.
    report.begin_page("4 · RQ2 — רשת, תוכן והטמעות")
    report.heading("RQ2 — רשת, תוכן וייצוגים נלמדים")
    rows: list[list[object]] = []
    for _, row in ablation.iterrows():
        rows.append(
            [
                str(row["model"]).replace("Logistic Regression - ", "LogReg "),
                _r(row["macro_f1_mean"]),
                _r(row["roc_auc_mean"]),
                _r(row["roc_auc_std"]),
            ]
        )
    for _, row in embeddings.iterrows():
        rows.append(
            [
                str(row["model"]).replace("Logistic Regression - ", "LogReg "),
                _r(row["macro_f1_mean"]),
                _r(row["roc_auc_mean"]),
                _r(row["roc_auc_std"]),
            ]
        )
    report.table(
        ["ייצוג / מקור מידע", "Macro F1", "ROC-AUC", "SD AUC"],
        rows,
        [295, 84, 84, CONTENT_W - 463],
        font_size=7.1,
        row_height=18,
    )
    report.figure(
        "representation_comparison.png",
        "איור 3. פיצ'רים ידניים, תוכן, DeepWalk, Node2Vec ושילובים.",
        box_height=275,
    )
    hybrid = ablation[ablation["model"].str.contains("Hybrid", regex=False)].iloc[0]
    content = ablation[
        ablation["model"].str.contains("Content Features Only", regex=False)
    ].iloc[0]
    network = ablation[
        ablation["model"].str.contains("Network Only", regex=False)
    ].iloc[0]
    report.paragraph(
        f"תוכן לבדו הגיע ל־AUC {content['roc_auc_mean']:.3f}, מול "
        f"{network['roc_auc_mean']:.3f} ברשת ידנית ו־{hybrid['roc_auc_mean']:.3f} "
        f"בשילוב. DeepWalk ו־Node2Vec לבדם הגיעו ל־{key['deepwalk_roc_auc']:.3f} "
        f"ו־{key['node2vec_roc_auc']:.3f}. הדמיון אינו מוכיח שקילות בין מקורות מידע."
    )
    report.paragraph(
        "מסקנת RQ2: התוכן חזק מן הייצוג הידני של הרשת, אך השילוב מוסיף ביצועים; "
        "הטמעות הליכות מרכזות מידע מבני שימושי ומצמצמות חלק מן הפער.",
        bold=True,
        size=8.6,
    )
    report.finish_page()

    # Page 6 — connected edge reconstruction.
    report.begin_page("5 · RQ3 — שחזור קשתות")
    report.heading("RQ3 — שחזור קשתות קשיר")
    report.paragraph(
        "בכל seed הוגנו קשתות עץ פורש, והוסרו 10% רק מן הקשתות האחרות. בכל חמש "
        "החלוקות גרף האימון נשאר רכיב קשיר יחיד וללא צמתים מבודדים. לכל positive "
        "הותאם non-edge מאותו סוג קצוות; Louvain וההטמעות חושבו על train בלבד."
    )
    recon_rows = [
        [
            row["heuristic"],
            _r(row["roc_auc_mean"]),
            _r(row["roc_auc_std"]),
            _r(row["average_precision_mean"]),
            _r(row["average_precision_std"]),
        ]
        for _, row in reconstruction.sort_values(
            "roc_auc_mean", ascending=False
        ).iterrows()
    ]
    report.table(
        ["שיטה", "ROC-AUC", "SD", "Average Precision", "SD"],
        recon_rows,
        [210, 77, 55, 130, CONTENT_W - 472],
        font_size=7.0,
        row_height=18,
    )
    report.figure(
        "edge_reconstruction_same_vs_cross.png",
        "איור 4. השוואה ישירה same-field מול cross-field בדגימה מאוזנת.",
        box_height=255,
    )
    best_group = recon_groups[
        recon_groups["heuristic"] == key["best_reconstruction_heuristic"]
    ].set_index("edge_group")
    same_auc = best_group.loc["same-field", "roc_auc_mean"]
    cross_auc = best_group.loc["cross-field", "roc_auc_mean"]
    same_ap = best_group.loc["same-field", "average_precision_mean"]
    cross_ap = best_group.loc["cross-field", "average_precision_mean"]
    report.paragraph(
        f"{key['best_reconstruction_heuristic']} הוביל עם AUC "
        f"{key['best_reconstruction_roc_auc']:.3f}. עבורו same-field קיבל AUC "
        f"{same_auc:.3f} ו־AP {same_ap:.3f}, לעומת AUC {cross_auc:.3f} ו־AP "
        f"{cross_ap:.3f} ב־cross-field. הפערים בכל השיטות קטנים ותלויי שיטה; "
        "הממצא מתאר שחזור קשתות שהוסתרו, לא התפתחות קשרים בזמן."
    )
    report.finish_page()

    # Page 7 — null and generative models.
    report.begin_page("6 · RQ4 — מודלי אפס ורשת")
    report.heading("RQ4 — מודלי אפס ומודלי רשת")
    report.table(
        ["מודל אפס", "נצפה", "ממוצע אפס", "SD", "p אמפירי"],
        [
            [
                "ערבוב תוויות",
                _r(null_h.loc["label_shuffle", "observed"]),
                _r(null_h.loc["label_shuffle", "null_mean"]),
                _r(null_h.loc["label_shuffle", "null_std"]),
                _r(null_h.loc["label_shuffle", "empirical_p"], 6),
            ],
            [
                "חיווט משמר־דרגה",
                _r(null_h.loc["degree_preserving_rewire", "observed"]),
                _r(null_h.loc["degree_preserving_rewire", "null_mean"]),
                _r(null_h.loc["degree_preserving_rewire", "null_std"]),
                _r(null_h.loc["degree_preserving_rewire", "empirical_p"], 6),
            ],
        ],
        [150, 76, 105, 66, CONTENT_W - 397],
        font_size=7.2,
        row_height=19,
    )
    report.figure(
        "homophily_null_distribution.png",
        "איור 5. הומופיליה נצפית מול ערבוב תוויות וחיווטים משמרי־דרגה.",
        box_height=235,
        gap=3,
    )
    network_rows = [
        [
            row["network"],
            f"{int(row['edges']):,}",
            _r(row["avg_clustering"], 4),
            _r(row["avg_shortest_path_sampled"]),
            int(row["max_degree"]),
        ]
        for _, row in networks.iterrows()
    ]
    report.table(
        ["רשת", "קשתות", "Clustering", "מרחק ממוצע", "דרגת שיא"],
        network_rows,
        [205, 82, 83, 92, CONTENT_W - 462],
        font_size=6.7,
        row_height=17,
    )
    report.paragraph(
        f"המודולריות הנצפית היא {null_m['observed_modularity']:.3f}, מול "
        f"{null_m['rewired_null_mean']:.3f} בחיווטים משמרי־דרגה "
        f"(p={null_m['empirical_p']:.6f}). התאמת Power Law נתנה "
        f"α={powerlaw['alpha']:.2f}; ההשוואה ללוג־נורמלית אינה מצדיקה הכרזה "
        "גורפת על scale-free. ER אינו משחזר התקבצות או דרגת שיא; BA משחזר חלק "
        "מן הזנב; Small-World משחזר התקבצות ומרחקים קצרים.",
        size=8.2,
        leading=10.2,
    )
    report.finish_page()

    # Page 8 — synthesis, limitations and references.
    report.begin_page("7 · דיון, מגבלות ומסקנות")
    report.heading("דיון מסכם ומגבלות")
    report.paragraph(
        "התמונה המצטברת עקבית: קיים קשר תיאורי חזק בין מבנה הרשת לתחום; קהילות "
        "והטמעות מוסיפות כוח חיזויי; תוכן מספק אות משלים. מודלי האפס מראים שרצף "
        "הדרגות מסביר חלק מן ההומופיליה אך לא את כולה. הצלחת Preferential "
        "Attachment תואמת את חשיבות הדרגות, אך אינה מזהה מנגנון סיבתי."
    )
    report.heading("מגבלות", size=11)
    for limitation in [
        "הנתונים הם תמונת מצב מיוני 2019 ואינם מייצגים בהכרח את GitHub כיום.",
        "תווית היעד בינארית ונגזרה ממידע פרופיל; ייתכנו רעש וחסר.",
        "הסיווג transductive ואינו מדמה צומת חדש ללא קשרים ידועים.",
        "Betweenness מחושב בקירוב וההטמעות נבדקו בתקציב קבוע ללא חיפוש רחב.",
        "שחזור קשתות שהוסתרו אינו תחזית זמן ואינו מזהה סיבתיות או כיווניות.",
        "מודלי האפס והמודלים הגנרטיביים מבודדים מאפיינים ואינם תהליך מלא.",
    ]:
        report.bullet(limitation)
    report.heading("מסקנות", size=11)
    report.paragraph(
        "RQ1: מבנה הרשת מאפשר סיווג מעל קווי הבסיס. RQ2: יש חפיפה חלקית ומידע "
        "משלים בין רשת לתוכן. RQ3: שחזור קשתות אפשרי תוך שמירת קישוריות, והפער "
        "same/cross תלוי בשיטה. RQ4: ההומופיליה והמודולריות חריגות ביחס למודלי "
        "האפס שנבדקו. הממצאים תומכים בערך האנליטי של רשתות מקצועיות, תוך שמירה "
        "על ניסוח תיאורי וזהיר."
    )
    report.heading("מקורות", size=11)
    references = [
        "1. Rozemberczki, Allen & Sarkar (2019), MUSAE.",
        "2. Blondel et al. (2008), Fast unfolding of communities.",
        "3. Perozzi, Al-Rfou & Skiena (2014), DeepWalk.",
        "4. Grover & Leskovec (2016), node2vec.",
        "5. Liben-Nowell & Kleinberg (2003), Link Prediction.",
        "6. Clauset, Shalizi & Newman (2009), Power-law distributions.",
        "7. Page et al. (1999), PageRank.",
        "8. מצגות ותרגולי הקורס: מרכזיות, קהילות, מודלי רשת, שחזור והטמעות.",
    ]
    for reference in references:
        report.paragraph(reference, size=7.4, leading=8.6, gap=0.2)
    report.save()
    print(f"Built final PDF: {REPORT_PATH}")




**פלט ומשמעות.** בלוק ``main`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

### build_report_pdf.py, שורות 730-731: `__main__ guard`

**מה ומדוע.** זהו שער ההרצה המקורי של קובץ CLI. במחברת הוא נשמר כלשונו, אך אינו מופעל כי `__name__` הוגדר כשם מחברת; ההרצה נעשית בתא מפורש ומתועד.

**שיטה/אלגוריתם.** הקוד נשמר בדיוק לפי המקור; ההסבר האלגוריתמי המלא ניתן בפרק המתודולוגי של המחברת.

**פלט צפוי.** אין פלט, משום ששער ה-CLI מושבת בכוונה בסביבת המחברת.

In [70]:
if __name__ == "__main__":
    main()


**פלט ומשמעות.** בלוק ``__main__ guard`` נטען בהצלחה. היעדר פלט הוא תקין עבור הגדרה; המשמעות המחקרית שלו נבחנת בתאי ההרצה והטבלאות שבהמשך, ולא מעצם הגדרת הפונקציה.

## אימות פלטי הפרויקט

התא מפעיל את מערכת ה-validation המקורית: קבצים נדרשים, CSV תקינים, גרף ופיצולי קשתות, p-values, קבוצות מודלים, מחברת legacy ודוח עד שמונה עמודים. מצופה לקבל `ALL FINAL PROJECT VALIDATIONS PASSED`.

In [71]:
_run_all_validations(ROOT, verify_notebook_execution=True)

ALL FINAL PROJECT VALIDATIONS PASSED


**פלט ומשמעות.** הודעת המעבר מתקבלת רק אם כל תנאי האיכות המקוריים מתקיימים; כשל מצביע על תוצר חסר או סתירה במספרים ולא מוסתר.

## הוכחת כיסוי byte-for-byte של כל `src/*.py`

לכל תא מקור נשמרים נתיב ושורות. התא מאחד את התאים לפי סדרם, משווה לטקסט המקורי ומוודא שכל 20 הקבצים מכוסים בדיוק פעם אחת בסדרה. זהו test אוטומטי לדרישת "ללא השמטות".

In [72]:
import nbformat
from collections import defaultdict

reconstructed = defaultdict(list)
series_paths = sorted((ROOT / "notebooks").glob("[0-9][0-9]_*.ipynb"))
for notebook_path in series_paths:
    notebook = nbformat.read(notebook_path, as_version=4)
    for cell in notebook.cells:
        source_file = cell.metadata.get("source_file")
        if source_file:
            reconstructed[source_file].append((cell.metadata["source_start_line"], cell.source))

coverage_rows = []
for source_path in sorted((ROOT / "src").glob("*.py")):
    relative = source_path.relative_to(ROOT).as_posix()
    combined = "".join(text for _, text in sorted(reconstructed.get(relative, [])))
    original = source_path.read_text(encoding="utf-8")
    coverage_rows.append({
        "source_file": relative,
        "source_lines": len(original.splitlines()),
        "covered_chars": len(combined),
        "source_chars": len(original),
        "exact_match": combined == original,
    })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
assert len(coverage_df) == 20
assert coverage_df["exact_match"].all(), coverage_df.loc[~coverage_df["exact_match"]]
print("ALL 20 SRC FILES ARE COVERED EXACTLY")

,source_file,source_lines,covered_chars,source_chars,exact_match
0,src/build_notebook.py,540,22812,22812,True
1,src/build_readme.py,258,8752,8752,True
2,src/build_report.py,843,32651,32651,True
3,src/build_report_pdf.py,731,26977,26977,True
4,src/community_analysis.py,148,4599,4599,True
5,src/content_features.py,125,4207,4207,True
6,src/data_loader.py,98,3815,3815,True
7,src/embeddings.py,176,5691,5691,True
8,src/final_validation.py,439,16390,16390,True
9,src/graph_analysis.py,123,4475,4475,True


ALL 20 SRC FILES ARE COVERED EXACTLY


**פלט ומשמעות.** כל שורה עם `exact_match=True` מוכיחה שהקוד הועבר בשלמותו בלי לשנות את קובצי המקור. הבדיקה משלימה את ה-validation הניסויי בבדיקת תיעוד וארגון.

## סיכום המחברת

**מה בוצע.** הופרדו הרחבות, הוצג כל קוד ה-build/validation, הורץ validation ונבדק כיסוי byte-for-byte.

**ממצאים מרכזיים.** ליבת הפרויקט משוחזרת ועקבית; ההרחבות מסומנות בבירור ואינן מנפחות את טענות הליבה.

**חיבור למחברת הבאה.** הסדרה מסתיימת כאן. להרצה חוזרת יש להשתמש בפקודות שב-notebooks/README.md לפי הסדר 01-09.